# 19. LaMa Report Generation

Refactored LaMa report-generation notebook for the standardized restoration universe.

Origin: existing notebook 16, updated from the old controlled 50-painting report into the current LaMa synthesis and handoff stage.


## Truth Source And Report Scope

This notebook is the final standalone LaMa synthesis report stage. It closes the LaMa-only evaluation block by consolidating restoration metadata, classical metrics, LPIPS metrics, feature-similarity metrics, diagnostic figures, validation outputs, and a downstream handoff manifest.

Required report evidence:

- consume the standardized LaMa restoration manifest;
- consume processed clean painting metadata;
- consume standardized LaMa classical, LPIPS, and feature-similarity metric outputs;
- use local damaged-region evidence for the main report tables: classical local region, LPIPS `mask_bbox_crop`, and feature-similarity `mask_bbox_crop`;
- exclude zero-control/empty-mask rows from the main diagnostic report dataframe because they do not define a damaged local crop;
- preserve zero-control/empty-mask rows as validation context through upstream handoff checks and report overview counts;
- include dataset, mask-type, painting-category, runtime/failure, correlation, and selected-case summaries;
- generate notebook plots and a standalone HTML report with interpretable numbers plus representative painting/difference-map figures;
- save a compact report dataframe, selected diagnostic cases, summary tables, validation CSV, stage manifest, and JSON handoff manifest.

Interpretation policy:

- this report summarizes LaMa behavior; it does not claim historical or semantic restoration correctness;
- metric disagreement is evidence, not noise, because pixel, perceptual, and feature-space metrics measure different restoration properties;
- selected cases are qualitative diagnostics and do not replace the complete metric tables;
- downstream notebooks should use this report handoff for comparison, grouping, and risk-flag stages.


## Batch Plan And File Contract

| Batch | Purpose | Main outputs |
|---|---|---|
| Batch 1 | Setup, truth source, imports, path contract, candidate resolution, and preflight validation | Batch 1 validation CSV and initial stage manifest |
| Batch 2 | Load inputs, verify upstream handoffs, inspect row/region/status contracts | Loaded source tables and upstream contract tables |
| Batch 3 | Build report dataframe and summary tables | `lama_report_dataframe.csv`, `lama_report_summaries.csv`, correlation/runtime summaries |
| Batch 4 | Select representative diagnostic cases and validate figure references | `lama_report_selected_cases.csv` and selected-case display tables |
| Batch 5 | Generate notebook plots and standalone HTML report | report figures, figure manifest, `lama_restoration_evaluation_report.html` |
| Batch 6 | Final validation, stage manifest, and downstream handoff manifest | final validation CSV, stage manifest, handoff manifest |

### Required Inputs

| Artifact | Standard path |
|---|---|
| Config | `config/experiment_50_config.yaml` |
| Processed clean metadata | `data/processed/metadata/metadata_processed_clean.csv` |
| LaMa restoration manifest | `data/processed/metadata/metadata_restored_lama.csv` |
| LaMa classical metrics | resolved from classical candidate paths in Batch 1 |
| LaMa LPIPS raw metrics | `outputs/metrics/lpips_metrics_lama.csv` |
| LaMa LPIPS compact metrics | `outputs/metrics/lpips_metrics_lama_compact.csv` |
| LaMa feature raw metrics | `outputs/metrics/feature_similarity_metrics_lama.csv` |
| LaMa feature compact metrics | `outputs/metrics/feature_similarity_metrics_lama_compact.csv` |
| LPIPS handoff manifest | `outputs/metrics/lpips_handoff_manifest_lama.json` |
| Feature handoff manifest | `outputs/metrics/feature_similarity_handoff_manifest_lama.json` |
| Error/difference-map manifest | resolved from error-map candidate paths in Batch 1 |
| Report helper module | `src/restoration_eval/reporting.py` |

### Planned Outputs

| Artifact | Standard path |
|---|---|
| Report dataframe | `outputs/metrics/lama_report_dataframe.csv` |
| Selected diagnostic cases | `outputs/metrics/lama_report_selected_cases.csv` |
| Report summaries | `outputs/metrics/lama_report_summaries.csv` |
| Metric correlation matrix | `outputs/metrics/lama_report_metric_correlations.csv` |
| Figure manifest | `outputs/figures/19_lama_report_generation/report_diagnostics_lama/lama_report_figure_manifest.csv` |
| HTML report | `outputs/reports/lama_restoration_evaluation_report.html` |
| Batch 1 validation | `outputs/validation/lama_report_generation_batch1_validation.csv` |
| Final validation | `outputs/validation/lama_report_generation_validation.csv` |
| Stage manifest | `outputs/manifests/19_lama_report_generation_manifest.json` |
| Handoff manifest | `outputs/metrics/lama_report_handoff_manifest.json` |


## Batch 1 - Setup, Contracts, And Input Gates

This batch establishes the notebook contract and checks whether the expected upstream LaMa artifacts are available. It does not build the report dataframe yet.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    yaml = None

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from restoration_eval.reporting import (
    MODEL_NAME_LAMA,
    REPORT_SCHEMA_VERSION,
    artifact_record,
    build_check,
    project_relative_path as reporting_project_relative_path,
    save_json,
    sha256_file,
)

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)


def project_relative_path(path: Path | str) -> str:
    return reporting_project_relative_path(path, project_root=PROJECT_ROOT)


def json_for_csv(value):
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, sort_keys=True, default=str)
    if isinstance(value, Path):
        return project_relative_path(value)
    return value


def path_audit_record(name: str, path: Path, *, required: bool = True) -> dict:
    exists = path.is_file()
    return {
        "artifact": name,
        "path": project_relative_path(path),
        "required": bool(required),
        "exists": bool(exists),
        "size_bytes": int(path.stat().st_size) if exists else 0,
    }

print("Project root:", PROJECT_ROOT)
print("Report helper schema:", REPORT_SCHEMA_VERSION)
print("Model:", MODEL_NAME_LAMA)


Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Report helper schema: 2.0.0
Model: lama


In [2]:

NOTEBOOK_ID = "19_lama_report_generation"
NOTEBOOK_NAME = "19_lama_report_generation.ipynb"
MODEL_NAME = MODEL_NAME_LAMA
REPORT_TITLE = "LaMa Restoration Evaluation Report"

CONFIG_PATH = PROJECT_ROOT / "config" / "experiment_50_config.yaml"
OUTPUTS_ROOT_DIR = PROJECT_ROOT / "outputs"

if CONFIG_PATH.is_file():
    if yaml is None:
        raise RuntimeError("PyYAML is required to read the experiment config.")

    with open(CONFIG_PATH, "r", encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)

    paths_cfg = config.get("paths", {})
    PROCESSED_METADATA_DIR = PROJECT_ROOT / paths_cfg.get(
        "processed_metadata_dir",
        "data/processed/metadata",
    )
    UPSTREAM_METRICS_DIR = PROJECT_ROOT / paths_cfg.get("metrics_dir", "outputs/metrics")
    UPSTREAM_REPORTS_DIR = PROJECT_ROOT / paths_cfg.get("reports_dir", "outputs/reports")
    UPSTREAM_FIGURES_ROOT_DIR = PROJECT_ROOT / paths_cfg.get("figures_dir", "outputs/figures")
else:
    config = {}
    PROCESSED_METADATA_DIR = PROJECT_ROOT / "data" / "processed" / "metadata"
    UPSTREAM_METRICS_DIR = OUTPUTS_ROOT_DIR / "metrics"
    UPSTREAM_REPORTS_DIR = OUTPUTS_ROOT_DIR / "reports"
    UPSTREAM_FIGURES_ROOT_DIR = OUTPUTS_ROOT_DIR / "figures"

# Notebook 19 owns all new report-generation artifacts in one folder. Upstream
# inputs remain where the earlier notebooks actually wrote them.
NOTEBOOK_OUTPUT_DIR = OUTPUTS_ROOT_DIR / NOTEBOOK_ID
METRICS_DIR = NOTEBOOK_OUTPUT_DIR / "metrics"
REPORTS_DIR = NOTEBOOK_OUTPUT_DIR / "reports"
VALIDATION_DIR = NOTEBOOK_OUTPUT_DIR / "validation"
MANIFESTS_DIR = NOTEBOOK_OUTPUT_DIR / "manifests"
FIGURES_DIR = NOTEBOOK_OUTPUT_DIR / "figures"
REPORT_DIAGNOSTIC_FIGURES_DIR = FIGURES_DIR / "report_diagnostics_lama"

DIFFERENCE_MAPS_DIR = OUTPUTS_ROOT_DIR / "16_lama_difference_maps"

for directory in [METRICS_DIR, REPORTS_DIR, VALIDATION_DIR, MANIFESTS_DIR, REPORT_DIAGNOSTIC_FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

HELPER_MODULE_PATH = SRC_DIR / "restoration_eval" / "reporting.py"
PROCESSED_CLEAN_METADATA_PATH = PROCESSED_METADATA_DIR / "metadata_processed_clean.csv"
LAMA_RESTORATION_MANIFEST_PATH = PROCESSED_METADATA_DIR / "metadata_restored_lama.csv"

CLASSICAL_METRICS_CANDIDATE_PATHS = [
    UPSTREAM_METRICS_DIR / "classical_metrics_lama.csv",
    UPSTREAM_METRICS_DIR / "classical_metrics_lama_compact.csv",
    UPSTREAM_METRICS_DIR / "classical_metrics_lama_full.csv",
    UPSTREAM_METRICS_DIR / "classical_metrics_lama_50.csv",
]

DIFFERENCE_MAP_MANIFEST_CANDIDATE_PATHS = [
    DIFFERENCE_MAPS_DIR / "lama_difference_map_manifest_all.csv",
    DIFFERENCE_MAPS_DIR / "lama_difference_map_manifest_selected.csv",
    UPSTREAM_METRICS_DIR / "difference_map_manifest_all_lama.csv",
    UPSTREAM_METRICS_DIR / "difference_map_manifest_lama.csv",
    UPSTREAM_METRICS_DIR / "error_map_manifest_all_lama.csv",
    UPSTREAM_METRICS_DIR / "error_map_manifest_lama.csv",
]

# Keep the old variable name because the reporting helper still uses
# error_map_figure_path internally for both error maps and difference maps.
ERROR_MAP_MANIFEST_CANDIDATE_PATHS = DIFFERENCE_MAP_MANIFEST_CANDIDATE_PATHS

OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH = (
    UPSTREAM_METRICS_DIR / "diagnostics" / "lama_diagnostic_panel_manifest.csv"
)

LPIPS_METRICS_PATH = UPSTREAM_METRICS_DIR / "lpips_metrics_lama.csv"
LPIPS_COMPACT_METRICS_PATH = UPSTREAM_METRICS_DIR / "lpips_metrics_lama_compact.csv"
FEATURE_METRICS_PATH = UPSTREAM_METRICS_DIR / "feature_similarity_metrics_lama.csv"
FEATURE_COMPACT_METRICS_PATH = UPSTREAM_METRICS_DIR / "feature_similarity_metrics_lama_compact.csv"
LPIPS_HANDOFF_MANIFEST_PATH = UPSTREAM_METRICS_DIR / "lpips_handoff_manifest_lama.json"
FEATURE_HANDOFF_MANIFEST_PATH = UPSTREAM_METRICS_DIR / "feature_similarity_handoff_manifest_lama.json"

REPORT_DATAFRAME_OUTPUT_PATH = METRICS_DIR / "lama_report_dataframe.csv"
SELECTED_CASES_OUTPUT_PATH = METRICS_DIR / "lama_report_selected_cases.csv"
REPORT_SUMMARIES_OUTPUT_PATH = METRICS_DIR / "lama_report_summaries.csv"
REPORT_CORRELATION_OUTPUT_PATH = METRICS_DIR / "lama_report_metric_correlations.csv"
REPORT_FIGURE_MANIFEST_PATH = REPORT_DIAGNOSTIC_FIGURES_DIR / "lama_report_figure_manifest.csv"
HTML_REPORT_OUTPUT_PATH = REPORTS_DIR / "lama_restoration_evaluation_report.html"
BATCH1_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch1_validation.csv"
FINAL_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_validation.csv"
STAGE_MANIFEST_JSON_PATH = MANIFESTS_DIR / "19_lama_report_generation_manifest.json"
HANDOFF_MANIFEST_JSON_PATH = METRICS_DIR / "lama_report_handoff_manifest.json"

EXPECTED_RESTORATION_CASES = 410
EXPECTED_REPORT_CASES = 355
EXPECTED_REPORT_REGION = "mask_bbox_crop"
EXPECTED_REPORT_STATUS = "ready_for_downstream_analysis"


In [3]:

def first_existing_path(candidate_paths: list[Path]) -> tuple[Path, bool, list[str]]:
    checked_paths = [project_relative_path(path) for path in candidate_paths]

    for path in candidate_paths:
        if path.is_file():
            return path, True, checked_paths

    return candidate_paths[0], False, checked_paths


CLASSICAL_METRICS_PATH, CLASSICAL_METRICS_FOUND, CLASSICAL_METRICS_CHECKED = first_existing_path(
    CLASSICAL_METRICS_CANDIDATE_PATHS
)
ERROR_MAP_MANIFEST_PATH, ERROR_MAP_MANIFEST_FOUND, ERROR_MAP_MANIFEST_CHECKED = first_existing_path(
    ERROR_MAP_MANIFEST_CANDIDATE_PATHS
)
DIFFERENCE_MAP_MANIFEST_PATH = ERROR_MAP_MANIFEST_PATH
DIFFERENCE_MAP_MANIFEST_FOUND = ERROR_MAP_MANIFEST_FOUND
DIFFERENCE_MAP_MANIFEST_CHECKED = ERROR_MAP_MANIFEST_CHECKED

REQUIRED_INPUT_PATHS = {
    "processed_clean_metadata": PROCESSED_CLEAN_METADATA_PATH,
    "lama_restoration_manifest": LAMA_RESTORATION_MANIFEST_PATH,
    "classical_metrics": CLASSICAL_METRICS_PATH,
    "lpips_raw_metrics": LPIPS_METRICS_PATH,
    "lpips_compact_metrics": LPIPS_COMPACT_METRICS_PATH,
    "feature_raw_metrics": FEATURE_METRICS_PATH,
    "feature_compact_metrics": FEATURE_COMPACT_METRICS_PATH,
    "lpips_handoff_manifest": LPIPS_HANDOFF_MANIFEST_PATH,
    "feature_handoff_manifest": FEATURE_HANDOFF_MANIFEST_PATH,
    "difference_map_manifest": DIFFERENCE_MAP_MANIFEST_PATH,
    "report_helper_module": HELPER_MODULE_PATH,
}

OPTIONAL_INPUT_PATHS = {
    "diagnostic_panel_manifest": OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH,
}

PLANNED_OUTPUT_PATHS = {
    "report_dataframe": REPORT_DATAFRAME_OUTPUT_PATH,
    "selected_cases": SELECTED_CASES_OUTPUT_PATH,
    "report_summaries": REPORT_SUMMARIES_OUTPUT_PATH,
    "report_metric_correlations": REPORT_CORRELATION_OUTPUT_PATH,
    "report_figure_manifest": REPORT_FIGURE_MANIFEST_PATH,
    "html_report": HTML_REPORT_OUTPUT_PATH,
    "batch1_validation": BATCH1_VALIDATION_OUTPUT_PATH,
    "final_validation": FINAL_VALIDATION_OUTPUT_PATH,
    "stage_manifest": STAGE_MANIFEST_JSON_PATH,
    "handoff_manifest": HANDOFF_MANIFEST_JSON_PATH,
}

input_contract_df = pd.DataFrame(
    [path_audit_record(name, path, required=True) for name, path in REQUIRED_INPUT_PATHS.items()]
    + [path_audit_record(name, path, required=False) for name, path in OPTIONAL_INPUT_PATHS.items()]
)

output_contract_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": project_relative_path(path),
            "parent_directory": project_relative_path(path.parent),
            "parent_exists": path.parent.is_dir(),
        }
        for name, path in PLANNED_OUTPUT_PATHS.items()
    ]
)

candidate_resolution_df = pd.DataFrame(
    [
        {
            "artifact": "classical_metrics",
            "resolved_path": project_relative_path(CLASSICAL_METRICS_PATH),
            "found": CLASSICAL_METRICS_FOUND,
            "checked_paths": CLASSICAL_METRICS_CHECKED,
        },
        {
            "artifact": "difference_map_manifest",
            "resolved_path": project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH),
            "found": DIFFERENCE_MAP_MANIFEST_FOUND,
            "checked_paths": DIFFERENCE_MAP_MANIFEST_CHECKED,
        },
        {
            "artifact": "diagnostic_panel_manifest_optional",
            "resolved_path": project_relative_path(OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH),
            "found": OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH.is_file(),
            "checked_paths": [project_relative_path(OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH)],
        },
    ]
)

print("Input contract:")
display(input_contract_df)

print("Output contract:")
display(output_contract_df)

print("Candidate path resolution:")
display(candidate_resolution_df)


Input contract:


,artifact,path,required,exists,size_bytes
0,processed_clean_metadata,data/processed/metadata/metadata_processed_clean.csv,True,True,30443
1,lama_restoration_manifest,data/processed/metadata/metadata_restored_lama.csv,True,True,1055479
2,classical_metrics,outputs/metrics/classical_metrics_lama.csv,True,True,2530737
3,lpips_raw_metrics,outputs/metrics/lpips_metrics_lama.csv,True,True,856763
4,lpips_compact_metrics,outputs/metrics/lpips_metrics_lama_compact.csv,True,True,526051
5,feature_raw_metrics,outputs/metrics/feature_similarity_metrics_lama.csv,True,True,1379909
6,feature_compact_metrics,outputs/metrics/feature_similarity_metrics_lama_compact.csv,True,True,1224285
7,lpips_handoff_manifest,outputs/metrics/lpips_handoff_manifest_lama.json,True,True,2222
8,feature_handoff_manifest,outputs/metrics/feature_similarity_handoff_manifest_lama.json,True,True,11164
9,difference_map_manifest,outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv,True,True,599615


Output contract:


,artifact,path,parent_directory,parent_exists
0,report_dataframe,outputs/19_lama_report_generation/metrics/lama_report_dataframe.csv,outputs/19_lama_report_generation/metrics,True
1,selected_cases,outputs/19_lama_report_generation/metrics/lama_report_selected_cases.csv,outputs/19_lama_report_generation/metrics,True
2,report_summaries,outputs/19_lama_report_generation/metrics/lama_report_summaries.csv,outputs/19_lama_report_generation/metrics,True
3,report_metric_correlations,outputs/19_lama_report_generation/metrics/lama_report_metric_correlations.csv,outputs/19_lama_report_generation/metrics,True
4,report_figure_manifest,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_figure_manifest.csv,outputs/19_lama_report_generation/figures/report_diagnostics_lama,True
5,html_report,outputs/19_lama_report_generation/reports/lama_restoration_evaluation_report.html,outputs/19_lama_report_generation/reports,True
6,batch1_validation,outputs/19_lama_report_generation/validation/lama_report_generation_batch1_validation.csv,outputs/19_lama_report_generation/validation,True
7,final_validation,outputs/19_lama_report_generation/validation/lama_report_generation_validation.csv,outputs/19_lama_report_generation/validation,True
8,stage_manifest,outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json,outputs/19_lama_report_generation/manifests,True
9,handoff_manifest,outputs/19_lama_report_generation/metrics/lama_report_handoff_manifest.json,outputs/19_lama_report_generation/metrics,True


Candidate path resolution:


,artifact,resolved_path,found,checked_paths
0,classical_metrics,outputs/metrics/classical_metrics_lama.csv,True,"[outputs/metrics/classical_metrics_lama.csv, outputs/metrics/classical_metrics_lama_compact.csv, outputs/metrics/classical_metrics_lama_full.csv, outputs/metrics/classical_metr..."
1,difference_map_manifest,outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv,True,"[outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv, outputs/16_lama_difference_maps/lama_difference_map_manifest_selected.csv, outputs/metrics/difference_map..."
2,diagnostic_panel_manifest_optional,outputs/metrics/diagnostics/lama_diagnostic_panel_manifest.csv,True,[outputs/metrics/diagnostics/lama_diagnostic_panel_manifest.csv]


In [4]:
def read_handoff_status(path: Path) -> str:
    if not path.is_file():
        return "missing"

    try:
        with open(path, "r", encoding="utf-8") as manifest_file:
            payload = json.load(manifest_file)
    except Exception as exc:
        return f"unreadable: {exc}"

    return str(payload.get("handoff_status", "missing_handoff_status"))


required_inputs_exist = {
    name: path.is_file()
    for name, path in REQUIRED_INPUT_PATHS.items()
}

required_inputs_nonempty = {
    name: int(path.stat().st_size) if path.is_file() else 0
    for name, path in REQUIRED_INPUT_PATHS.items()
}

lpips_handoff_status = read_handoff_status(LPIPS_HANDOFF_MANIFEST_PATH)
feature_handoff_status = read_handoff_status(FEATURE_HANDOFF_MANIFEST_PATH)
helper_sha256 = sha256_file(HELPER_MODULE_PATH) if HELPER_MODULE_PATH.is_file() else ""

batch1_validation_df = pd.DataFrame(
    [
        build_check(
            "report_helper_exists",
            HELPER_MODULE_PATH.is_file(),
            True,
            HELPER_MODULE_PATH.is_file(),
            "reporting helper module is missing",
        ),
        build_check(
            "report_helper_nonempty",
            HELPER_MODULE_PATH.stat().st_size if HELPER_MODULE_PATH.is_file() else 0,
            "> 0",
            HELPER_MODULE_PATH.is_file() and HELPER_MODULE_PATH.stat().st_size > 0,
            "reporting helper module is empty",
        ),
        build_check(
            "classical_metrics_candidate_resolved",
            {
                "resolved_path": project_relative_path(CLASSICAL_METRICS_PATH),
                "checked_paths": CLASSICAL_METRICS_CHECKED,
            },
            "one existing classical metrics path",
            CLASSICAL_METRICS_FOUND,
            "none of the classical metrics candidate paths exists",
        ),
        build_check(
            "error_map_manifest_candidate_resolved",
            {
                "resolved_path": project_relative_path(ERROR_MAP_MANIFEST_PATH),
                "checked_paths": ERROR_MAP_MANIFEST_CHECKED,
            },
            "one existing error/difference-map manifest path",
            ERROR_MAP_MANIFEST_FOUND,
            "none of the error/difference-map manifest candidate paths exists",
        ),
        build_check(
            "required_inputs_exist",
            required_inputs_exist,
            "all true",
            all(required_inputs_exist.values()),
            "one or more required input artifacts is missing",
        ),
        build_check(
            "required_inputs_nonempty",
            required_inputs_nonempty,
            "all > 0",
            all(value > 0 for value in required_inputs_nonempty.values()),
            "one or more required input artifacts is empty",
        ),
        build_check(
            "lpips_handoff_ready",
            lpips_handoff_status,
            EXPECTED_REPORT_STATUS,
            lpips_handoff_status == EXPECTED_REPORT_STATUS,
            "LPIPS handoff manifest is not ready",
        ),
        build_check(
            "feature_handoff_ready",
            feature_handoff_status,
            EXPECTED_REPORT_STATUS,
            feature_handoff_status == EXPECTED_REPORT_STATUS,
            "feature-similarity handoff manifest is not ready",
        ),
        build_check(
            "planned_output_directories_exist",
            {name: path.parent.is_dir() for name, path in PLANNED_OUTPUT_PATHS.items()},
            "all true",
            all(path.parent.is_dir() for path in PLANNED_OUTPUT_PATHS.values()),
            "one or more output directories could not be created",
        ),
    ]
)

batch1_validation_export_df = batch1_validation_df.copy()
for column in ["observed", "expected"]:
    batch1_validation_export_df[column] = batch1_validation_export_df[column].map(json_for_csv)

BATCH1_VALIDATION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
batch1_validation_export_df.to_csv(BATCH1_VALIDATION_OUTPUT_PATH, index=False)

stage_manifest = {
    "manifest_type": "lama_report_generation_stage",
    "notebook": {
        "id": NOTEBOOK_ID,
        "name": NOTEBOOK_NAME,
        "model_name": MODEL_NAME,
        "report_schema_version": REPORT_SCHEMA_VERSION,
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "truth_source_scope": {
        "purpose": "final standalone LaMa synthesis report and downstream handoff",
        "main_report_region": EXPECTED_REPORT_REGION,
        "expected_restoration_cases": EXPECTED_RESTORATION_CASES,
        "expected_report_cases": EXPECTED_REPORT_CASES,
        "zero_controls_in_main_report": False,
    },
    "input_paths": {
        name: project_relative_path(path)
        for name, path in REQUIRED_INPUT_PATHS.items()
    },
    "candidate_resolution": candidate_resolution_df.to_dict(orient="records"),
    "planned_outputs": {
        name: project_relative_path(path)
        for name, path in PLANNED_OUTPUT_PATHS.items()
    },
    "planned_batches": {
        "batch_1": "setup_contracts_and_input_gates",
        "batch_2": "load_inputs_and_verify_upstream_contracts",
        "batch_3": "build_report_dataframe_and_summaries",
        "batch_4": "select_diagnostic_cases_and_validate_figures",
        "batch_5": "generate_plots_and_html_report",
        "batch_6": "final_validation_and_handoff_manifest",
    },
    "batches": {
        "batch_1": {
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "purpose": "setup_contracts_and_input_gates",
            "validation": project_relative_path(BATCH1_VALIDATION_OUTPUT_PATH),
            "validation_passed": bool(batch1_validation_df["passed"].astype(bool).all()),
            "helper_sha256": helper_sha256,
        }
    },
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

display(batch1_validation_df)

print("Batch 1 validation:", project_relative_path(BATCH1_VALIDATION_OUTPUT_PATH))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))
print("Helper SHA256:", helper_sha256)

if not batch1_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Batch 1 preflight failed. Review the validation table above before continuing.")

print("Batch 1 complete. Continue with Batch 2 after confirming these paths are correct.")


,check_name,observed,expected,passed,issue
0,report_helper_exists,True,True,True,
1,report_helper_nonempty,53240,> 0,True,
2,classical_metrics_candidate_resolved,"{'resolved_path': 'outputs/metrics/classical_metrics_lama.csv', 'checked_paths': ['outputs/metrics/classical_metrics_lama.csv', 'outputs/metrics/classical_metrics_lama_compact....",one existing classical metrics path,True,
3,error_map_manifest_candidate_resolved,"{'resolved_path': 'outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv', 'checked_paths': ['outputs/16_lama_difference_maps/lama_difference_map_manifest_all.cs...",one existing error/difference-map manifest path,True,
4,required_inputs_exist,"{'processed_clean_metadata': True, 'lama_restoration_manifest': True, 'classical_metrics': True, 'lpips_raw_metrics': True, 'lpips_compact_metrics': True, 'feature_raw_metrics'...",all true,True,
5,required_inputs_nonempty,"{'processed_clean_metadata': 30443, 'lama_restoration_manifest': 1055479, 'classical_metrics': 2530737, 'lpips_raw_metrics': 856763, 'lpips_compact_metrics': 526051, 'feature_r...",all > 0,True,
6,lpips_handoff_ready,ready_for_downstream_analysis,ready_for_downstream_analysis,True,
7,feature_handoff_ready,ready_for_downstream_analysis,ready_for_downstream_analysis,True,
8,planned_output_directories_exist,"{'report_dataframe': True, 'selected_cases': True, 'report_summaries': True, 'report_metric_correlations': True, 'report_figure_manifest': True, 'html_report': True, 'batch1_va...",all true,True,


Batch 1 validation: outputs/19_lama_report_generation/validation/lama_report_generation_batch1_validation.csv
Stage manifest: outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json
Helper SHA256: 97c7655aa321066f74c2130eac4afce6dffd342d9ff837d8bc7270e79fe6577a
Batch 1 complete. Continue with Batch 2 after confirming these paths are correct.


In [5]:
# ============================================================
# Batch 2 - Load Inputs And Verify Upstream Contracts
# ============================================================

BATCH2_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch2_validation.csv"

print("Batch 2 target:")
print("Load inputs, verify upstream handoff statuses, validate row/region/status contracts.")

Batch 2 target:
Load inputs, verify upstream handoff statuses, validate row/region/status contracts.


In [6]:
from restoration_eval.reporting import (
    CLASSICAL_REPORT_REGION_PREFERENCE,
    FEATURE_REPORT_REGION,
    LPIPS_REPORT_REGION,
    REPORT_REQUIRED_METADATA_COLUMNS,
    REPORT_REQUIRED_RESTORATION_COLUMNS,
    add_zero_control_flag,
    first_existing_column,
    require_columns,
)

EXPECTED_METRIC_REGION_COUNTS = {
    "full_image": EXPECTED_RESTORATION_CASES,
    "content_region": EXPECTED_RESTORATION_CASES,
    EXPECTED_REPORT_REGION: EXPECTED_REPORT_CASES,
}

EXPECTED_COMPLETED_UPSTREAM_BATCHES = {
    "lpips": [f"batch_{batch_number}" for batch_number in range(1, 7)],
    "feature_similarity": [f"batch_{batch_number}" for batch_number in range(1, 7)],
}


def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as json_file:
        return json.load(json_file)


def validation_passed_from_csv(path: Path) -> bool:
    if not path.is_file():
        return False

    validation_df = pd.read_csv(path)

    if "passed" not in validation_df.columns:
        return False

    return bool(validation_df["passed"].astype(bool).all())


def safe_read_csv(path: Path, *, dataframe_name: str) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"{dataframe_name} not found: {project_relative_path(path)}")

    dataframe = pd.read_csv(path, low_memory=False)

    if dataframe.empty:
        raise ValueError(f"{dataframe_name} is empty: {project_relative_path(path)}")

    return dataframe


def normalise_case_id_for_contract(dataframe: pd.DataFrame) -> pd.DataFrame:
    dataframe = dataframe.copy()

    if "case_id" not in dataframe.columns:
        for candidate_column in [
            "restoration_case_id",
            "metric_case_id",
            "source_case_id",
            "feature_case_id",
            "lpips_case_id",
        ]:
            if candidate_column in dataframe.columns:
                dataframe["case_id"] = dataframe[candidate_column]
                break

    if "case_id" in dataframe.columns:
        dataframe["case_id"] = dataframe["case_id"].astype(str)

    return dataframe


def metric_region_counts(dataframe: pd.DataFrame) -> dict:
    if "evaluation_region" not in dataframe.columns:
        return {}

    return {
        str(key): int(value)
        for key, value in dataframe["evaluation_region"].astype(str).value_counts().to_dict().items()
    }


def metric_status_counts(dataframe: pd.DataFrame) -> dict:
    if "status" not in dataframe.columns:
        return {}

    return {
        str(key): int(value)
        for key, value in dataframe["status"].astype(str).value_counts(dropna=False).to_dict().items()
    }


def region_case_ids(dataframe: pd.DataFrame, region: str) -> set[str]:
    dataframe = normalise_case_id_for_contract(dataframe)

    if "evaluation_region" not in dataframe.columns or "case_id" not in dataframe.columns:
        return set()

    region_mask = dataframe["evaluation_region"].astype(str).eq(region)

    if "status" in dataframe.columns:
        region_mask &= dataframe["status"].astype(str).eq("ok")

    return set(dataframe.loc[region_mask, "case_id"].dropna().astype(str))


def duplicate_case_region_rows(dataframe: pd.DataFrame) -> int:
    dataframe = normalise_case_id_for_contract(dataframe)

    if not {"case_id", "evaluation_region"}.issubset(dataframe.columns):
        return -1

    return int(dataframe.duplicated(["case_id", "evaluation_region"]).sum())


def get_manifest_batch_status(manifest_payload: dict, batch_names: list[str]) -> dict:
    batches = manifest_payload.get("batches", {})

    return {
        batch_name: bool(batches.get(batch_name, {}).get("validation_passed", False))
        for batch_name in batch_names
    }


def get_manifest_raw_contract(manifest_payload: dict) -> dict:
    return (
        manifest_payload
        .get("contracts", {})
        .get("raw_metric_contract", {})
    )


def dataframe_shape_record(name: str, dataframe: pd.DataFrame) -> dict:
    return {
        "table": name,
        "rows": int(len(dataframe)),
        "columns": int(len(dataframe.columns)),
    }

FIGURE_PATH_COLUMN_CANDIDATES = [
    "difference_map_figure_path",
    "difference_map_path",
    "difference_map_file_path",
    "difference_map_file",
    "error_map_figure_path",
    "panel_path",
    "diagnostic_panel_path",
    "diagnostic_panel_figure_path",
    "figure_path",
    "output_path",
    "output_figure_path",
    "saved_figure_path",
    "figure_file_path",
    "relative_path",
    "path",
]


def resolve_project_path_from_manifest(value) -> Path | None:
    if value is None or pd.isna(value):
        return None

    value_text = str(value).strip().strip('"').strip("'")
    if not value_text:
        return None

    normalised_text = value_text.replace("\\", "/")
    candidates = []

    for candidate_text in [value_text, normalised_text]:
        candidate_path = Path(candidate_text)
        if candidate_path.is_absolute():
            candidates.append(candidate_path)
        candidates.append(PROJECT_ROOT / candidate_path)

    lower_text = normalised_text.lower()
    for marker in ["outputs/", "data/", "notebooks/", "src/", "config/"]:
        marker_index = lower_text.find(marker)
        if marker_index >= 0:
            candidates.append(PROJECT_ROOT / normalised_text[marker_index:])

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        candidate_key = str(candidate)
        if candidate_key not in seen:
            seen.add(candidate_key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.is_file():
            return candidate

    # Return the most portable project-relative interpretation when the file is
    # not visible from the current runtime yet; later checks will report missing files.
    for candidate in unique_candidates:
        if str(candidate).startswith(str(PROJECT_ROOT)):
            return candidate

    return unique_candidates[0] if unique_candidates else None


def prepare_notebook_figure_manifest_paths(
    figure_manifest_df: pd.DataFrame,
    *,
    key_column: str = "case_id",
    output_column: str = "error_map_figure_path",
) -> tuple[pd.DataFrame, str | None]:
    manifest_df = normalise_case_id_for_contract(figure_manifest_df)
    require_columns(manifest_df, [key_column], dataframe_name="difference_map_manifest_df")

    path_column = first_existing_column(manifest_df, FIGURE_PATH_COLUMN_CANDIDATES)

    if path_column is None:
        return pd.DataFrame(columns=[key_column, output_column]), None

    context_columns = [
        column
        for column in [
            key_column,
            path_column,
            "painting_id",
            "dataset_name",
            "mask_type",
            "diagnostic_reason",
            "selection_reason",
            "status",
            "figure_type",
            "panel_size_mb",
        ]
        if column in manifest_df.columns
    ]

    prepared_df = manifest_df[context_columns].copy()
    prepared_df = prepared_df.rename(columns={path_column: output_column})
    prepared_df[output_column] = prepared_df[output_column].map(resolve_project_path_from_manifest)
    prepared_df = prepared_df.drop_duplicates(subset=[key_column]).reset_index(drop=True)

    return prepared_df, path_column


In [7]:
processed_metadata_df = safe_read_csv(
    PROCESSED_CLEAN_METADATA_PATH,
    dataframe_name="processed clean metadata",
)

restored_metadata_df = safe_read_csv(
    LAMA_RESTORATION_MANIFEST_PATH,
    dataframe_name="LaMa restoration manifest",
)

classical_metrics_df = safe_read_csv(
    CLASSICAL_METRICS_PATH,
    dataframe_name="LaMa classical metrics",
)

lpips_raw_metrics_df = safe_read_csv(
    LPIPS_METRICS_PATH,
    dataframe_name="LaMa LPIPS raw metrics",
)

lpips_compact_metrics_df = safe_read_csv(
    LPIPS_COMPACT_METRICS_PATH,
    dataframe_name="LaMa LPIPS compact metrics",
)

feature_raw_metrics_df = safe_read_csv(
    FEATURE_METRICS_PATH,
    dataframe_name="LaMa feature-similarity raw metrics",
)

feature_compact_metrics_df = safe_read_csv(
    FEATURE_COMPACT_METRICS_PATH,
    dataframe_name="LaMa feature-similarity compact metrics",
)

error_map_manifest_df = safe_read_csv(
    DIFFERENCE_MAP_MANIFEST_PATH,
    dataframe_name="LaMa difference-map manifest",
)

lpips_handoff_manifest = read_json(LPIPS_HANDOFF_MANIFEST_PATH)
feature_handoff_manifest = read_json(FEATURE_HANDOFF_MANIFEST_PATH)

# Batch 3 helper expects these canonical names.
lpips_metrics_df = lpips_raw_metrics_df.copy()
feature_metrics_df = feature_raw_metrics_df.copy()

restored_metadata_df = normalise_case_id_for_contract(restored_metadata_df)
classical_metrics_df = normalise_case_id_for_contract(classical_metrics_df)
lpips_raw_metrics_df = normalise_case_id_for_contract(lpips_raw_metrics_df)
lpips_compact_metrics_df = normalise_case_id_for_contract(lpips_compact_metrics_df)
feature_raw_metrics_df = normalise_case_id_for_contract(feature_raw_metrics_df)
feature_compact_metrics_df = normalise_case_id_for_contract(feature_compact_metrics_df)
error_map_manifest_df = normalise_case_id_for_contract(error_map_manifest_df)

restored_metadata_df = add_zero_control_flag(restored_metadata_df)

loaded_table_shapes_df = pd.DataFrame(
    [
        dataframe_shape_record("processed_metadata", processed_metadata_df),
        dataframe_shape_record("restored_metadata", restored_metadata_df),
        dataframe_shape_record("classical_metrics", classical_metrics_df),
        dataframe_shape_record("lpips_raw_metrics", lpips_raw_metrics_df),
        dataframe_shape_record("lpips_compact_metrics", lpips_compact_metrics_df),
        dataframe_shape_record("feature_raw_metrics", feature_raw_metrics_df),
        dataframe_shape_record("feature_compact_metrics", feature_compact_metrics_df),
        dataframe_shape_record("difference_map_manifest", error_map_manifest_df),
    ]
)

upstream_handoff_df = pd.DataFrame(
    [
        {
            "upstream": "lpips",
            "path": project_relative_path(LPIPS_HANDOFF_MANIFEST_PATH),
            "handoff_status": lpips_handoff_manifest.get("handoff_status", ""),
            "validation_passed": bool(
                lpips_handoff_manifest.get("validation", {}).get("final_validation_passed", False)
            ),
            "raw_contract": get_manifest_raw_contract(lpips_handoff_manifest),
        },
        {
            "upstream": "feature_similarity",
            "path": project_relative_path(FEATURE_HANDOFF_MANIFEST_PATH),
            "handoff_status": feature_handoff_manifest.get("handoff_status", ""),
            "validation_passed": bool(
                feature_handoff_manifest.get("validation", {}).get("final_validation_passed", False)
            ),
            "raw_contract": get_manifest_raw_contract(feature_handoff_manifest),
        },
    ]
)

print("Loaded table shapes:")
display(loaded_table_shapes_df)

print("Upstream handoff manifests:")
display(upstream_handoff_df)


Loaded table shapes:


,table,rows,columns
0,processed_metadata,50,47
1,restored_metadata,410,219
2,classical_metrics,2260,51
3,lpips_raw_metrics,1175,47
4,lpips_compact_metrics,1175,25
5,feature_raw_metrics,1175,59
6,feature_compact_metrics,1175,48
7,difference_map_manifest,360,109


Upstream handoff manifests:


,upstream,path,handoff_status,validation_passed,raw_contract
0,lpips,outputs/metrics/lpips_handoff_manifest_lama.json,ready_for_downstream_analysis,False,{}
1,feature_similarity,outputs/metrics/feature_similarity_handoff_manifest_lama.json,ready_for_downstream_analysis,True,"{'cases': 410, 'expected_rows': 1175, 'actual_rows': 1175, 'expected_region_counts': {'full_image': 410, 'content_region': 410, 'mask_bbox_crop': 355}, 'actual_region_counts': ..."


In [8]:
require_columns(
    processed_metadata_df,
    REPORT_REQUIRED_METADATA_COLUMNS,
    dataframe_name="processed_metadata_df",
)

require_columns(
    restored_metadata_df,
    REPORT_REQUIRED_RESTORATION_COLUMNS,
    dataframe_name="restored_metadata_df",
)

for dataframe_name, dataframe in [
    ("classical_metrics_df", classical_metrics_df),
    ("lpips_raw_metrics_df", lpips_raw_metrics_df),
    ("lpips_compact_metrics_df", lpips_compact_metrics_df),
    ("feature_raw_metrics_df", feature_raw_metrics_df),
    ("feature_compact_metrics_df", feature_compact_metrics_df),
]:
    require_columns(
        dataframe,
        ["case_id", "evaluation_region"],
        dataframe_name=dataframe_name,
    )

restored_lama_df = restored_metadata_df.loc[
    restored_metadata_df["model_name"].astype(str).eq(MODEL_NAME)
].copy()

if "status" in restored_lama_df.columns:
    restored_lama_ok_df = restored_lama_df.loc[
        restored_lama_df["status"].astype(str).eq("ok")
    ].copy()
else:
    restored_lama_ok_df = restored_lama_df.copy()

report_restoration_df = restored_lama_ok_df.loc[
    ~restored_lama_ok_df["is_zero_control"].astype(bool)
].copy()

report_case_ids = set(report_restoration_df["case_id"].dropna().astype(str))
zero_control_case_ids = set(
    restored_lama_ok_df.loc[
        restored_lama_ok_df["is_zero_control"].astype(bool),
        "case_id",
    ].dropna().astype(str)
)

classical_available_regions = sorted(
    classical_metrics_df["evaluation_region"].dropna().astype(str).unique().tolist()
)

classical_selected_region = next(
    (
        region
        for region in CLASSICAL_REPORT_REGION_PREFERENCE
        if region in classical_available_regions
    ),
    "",
)

classical_report_case_ids = region_case_ids(classical_metrics_df, classical_selected_region)
lpips_report_case_ids = region_case_ids(lpips_raw_metrics_df, LPIPS_REPORT_REGION)
feature_report_case_ids = region_case_ids(feature_raw_metrics_df, FEATURE_REPORT_REGION)

difference_map_path_column = first_existing_column(
    error_map_manifest_df,
    FIGURE_PATH_COLUMN_CANDIDATES,
)

# Backward-compatible alias for the old helper/report naming.
diagnostic_path_column = difference_map_path_column

difference_map_case_column_present = "case_id" in error_map_manifest_df.columns

# Backward-compatible alias for the old helper/report naming.
diagnostic_case_column_present = difference_map_case_column_present

lpips_stage_batch_status = get_manifest_batch_status(
    lpips_handoff_manifest,
    EXPECTED_COMPLETED_UPSTREAM_BATCHES["lpips"],
)

feature_stage_batch_status = get_manifest_batch_status(
    feature_handoff_manifest,
    EXPECTED_COMPLETED_UPSTREAM_BATCHES["feature_similarity"],
)

region_status_contract_df = pd.DataFrame(
    [
        {
            "table": "classical_metrics",
            "rows": int(len(classical_metrics_df)),
            "regions": metric_region_counts(classical_metrics_df),
            "status_counts": metric_status_counts(classical_metrics_df),
            "duplicate_case_region_rows": duplicate_case_region_rows(classical_metrics_df),
            "selected_report_region": classical_selected_region,
            "selected_report_region_cases": int(len(classical_report_case_ids)),
        },
        {
            "table": "lpips_raw_metrics",
            "rows": int(len(lpips_raw_metrics_df)),
            "regions": metric_region_counts(lpips_raw_metrics_df),
            "status_counts": metric_status_counts(lpips_raw_metrics_df),
            "duplicate_case_region_rows": duplicate_case_region_rows(lpips_raw_metrics_df),
            "selected_report_region": LPIPS_REPORT_REGION,
            "selected_report_region_cases": int(len(lpips_report_case_ids)),
        },
        {
            "table": "lpips_compact_metrics",
            "rows": int(len(lpips_compact_metrics_df)),
            "regions": metric_region_counts(lpips_compact_metrics_df),
            "status_counts": metric_status_counts(lpips_compact_metrics_df),
            "duplicate_case_region_rows": duplicate_case_region_rows(lpips_compact_metrics_df),
            "selected_report_region": LPIPS_REPORT_REGION,
            "selected_report_region_cases": int(len(region_case_ids(lpips_compact_metrics_df, LPIPS_REPORT_REGION))),
        },
        {
            "table": "feature_raw_metrics",
            "rows": int(len(feature_raw_metrics_df)),
            "regions": metric_region_counts(feature_raw_metrics_df),
            "status_counts": metric_status_counts(feature_raw_metrics_df),
            "duplicate_case_region_rows": duplicate_case_region_rows(feature_raw_metrics_df),
            "selected_report_region": FEATURE_REPORT_REGION,
            "selected_report_region_cases": int(len(feature_report_case_ids)),
        },
        {
            "table": "feature_compact_metrics",
            "rows": int(len(feature_compact_metrics_df)),
            "regions": metric_region_counts(feature_compact_metrics_df),
            "status_counts": metric_status_counts(feature_compact_metrics_df),
            "duplicate_case_region_rows": duplicate_case_region_rows(feature_compact_metrics_df),
            "selected_report_region": FEATURE_REPORT_REGION,
            "selected_report_region_cases": int(len(region_case_ids(feature_compact_metrics_df, FEATURE_REPORT_REGION))),
        },
    ]
)

print("Restoration row counts:")
display(
    pd.DataFrame(
        [
            {"item": "LaMa restoration rows", "value": int(len(restored_lama_df))},
            {"item": "LaMa ok restoration rows", "value": int(len(restored_lama_ok_df))},
            {"item": "Main report non-zero rows", "value": int(len(report_restoration_df))},
            {"item": "Zero-control/empty-mask rows", "value": int(len(zero_control_case_ids))},
            {"item": "Unique main report case IDs", "value": int(len(report_case_ids))},
        ]
    )
)

print("Metric region/status contract:")
display(region_status_contract_df)

print("Difference-map manifest path contract:")
display(
    pd.DataFrame(
        [
            {
                "manifest_path": project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH),
                "rows": int(len(error_map_manifest_df)),
                "has_case_id": difference_map_case_column_present,
                "path_column": difference_map_path_column or "",
                "columns": list(error_map_manifest_df.columns),
            }
        ]
    )
)


Restoration row counts:


,item,value
0,LaMa restoration rows,410
1,LaMa ok restoration rows,410
2,Main report non-zero rows,355
3,Zero-control/empty-mask rows,55
4,Unique main report case IDs,355


Metric region/status contract:


,table,rows,regions,status_counts,duplicate_case_region_rows,selected_report_region,selected_report_region_cases
0,classical_metrics,2260,"{'full_image': 410, 'content_region': 410, 'masked_region': 360, 'boundary_region': 360, 'outside_mask_region': 360, 'mask_bbox_crop': 360}",{'ok': 2260},0,masked_region,360
1,lpips_raw_metrics,1175,"{'full_image': 410, 'content_region': 410, 'mask_bbox_crop': 355}",{'ok': 1175},0,mask_bbox_crop,355
2,lpips_compact_metrics,1175,"{'content_region': 410, 'full_image': 410, 'mask_bbox_crop': 355}",{'ok': 1175},0,mask_bbox_crop,355
3,feature_raw_metrics,1175,"{'full_image': 410, 'content_region': 410, 'mask_bbox_crop': 355}",{'ok': 1175},0,mask_bbox_crop,355
4,feature_compact_metrics,1175,"{'full_image': 410, 'content_region': 410, 'mask_bbox_crop': 355}",{'ok': 1175},0,mask_bbox_crop,355


Difference-map manifest path contract:


,manifest_path,rows,has_case_id,path_column,columns
0,outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv,360,True,figure_path,"[notebook, spatial_schema_version, implementation, restoration_case_id, case_id, source_case_id, dataset_name, painting_id, category, title, mask_id, mask_type, model_name, sel..."


In [9]:

# ============================================================
# Batch 2 Fix - Current Notebook 16 Difference-Map Manifest Semantics
# ============================================================

# Notebook 16 is the canonical source for LaMa difference-map figures. The older
# 12-row diagnostic panel manifest is optional and must not drive report coverage.
difference_map_path_column = first_existing_column(
    error_map_manifest_df,
    FIGURE_PATH_COLUMN_CANDIDATES,
)
diagnostic_path_column = difference_map_path_column

# Classical metrics contains 5 extra masked-region cases. That is okay as long
# as all 355 report cases are covered; later joins filter to report_case_ids.
classical_report_case_ids = classical_report_case_ids & report_case_ids

# LPIPS/feature handoff manifests do not consistently expose per-stage batch status.
# The authoritative gates for this report are the handoff status plus row/region/status checks below.
lpips_stage_batch_status = {
    "handoff_ready": lpips_handoff_manifest.get("handoff_status", "") == EXPECTED_REPORT_STATUS,
    "row_region_status_contract_checked_below": True,
}

feature_stage_batch_status = {
    "handoff_ready": feature_handoff_manifest.get("handoff_status", "") == EXPECTED_REPORT_STATUS,
    "row_region_status_contract_checked_below": True,
}

print("Difference-map manifest rows:", len(error_map_manifest_df))
print("Difference-map path column:", difference_map_path_column)
print("Filtered classical report cases:", len(classical_report_case_ids))
print("LPIPS effective stage gate:", lpips_stage_batch_status)
print("Feature effective stage gate:", feature_stage_batch_status)


Difference-map manifest rows: 360
Difference-map path column: figure_path
Filtered classical report cases: 355
LPIPS effective stage gate: {'handoff_ready': True, 'row_region_status_contract_checked_below': True}
Feature effective stage gate: {'handoff_ready': True, 'row_region_status_contract_checked_below': True}


In [10]:
lpips_raw_contract = get_manifest_raw_contract(lpips_handoff_manifest)
feature_raw_contract = get_manifest_raw_contract(feature_handoff_manifest)

lpips_raw_expected_rows = lpips_raw_contract.get("expected_rows", EXPECTED_RESTORATION_CASES * 2 + EXPECTED_REPORT_CASES)
feature_raw_expected_rows = feature_raw_contract.get("expected_rows", EXPECTED_RESTORATION_CASES * 2 + EXPECTED_REPORT_CASES)

lpips_raw_expected_region_counts = {
    str(key): int(value)
    for key, value in lpips_raw_contract.get("expected_region_counts", EXPECTED_METRIC_REGION_COUNTS).items()
}

feature_raw_expected_region_counts = {
    str(key): int(value)
    for key, value in feature_raw_contract.get("expected_region_counts", EXPECTED_METRIC_REGION_COUNTS).items()
}

batch2_validation_df = pd.DataFrame(
    [
        build_check(
            "batch1_validation_passed",
            validation_passed_from_csv(BATCH1_VALIDATION_OUTPUT_PATH),
            True,
            validation_passed_from_csv(BATCH1_VALIDATION_OUTPUT_PATH),
            "Batch 1 validation did not pass",
        ),
        build_check(
            "loaded_tables_nonempty",
            {
                row["table"]: int(row["rows"])
                for _, row in loaded_table_shapes_df.iterrows()
            },
            "all > 0",
            bool((loaded_table_shapes_df["rows"].astype(int) > 0).all()),
            "one or more loaded tables is empty",
        ),
        build_check(
            "processed_metadata_required_columns_present",
            [
                column
                for column in REPORT_REQUIRED_METADATA_COLUMNS
                if column not in processed_metadata_df.columns
            ],
            [],
            all(column in processed_metadata_df.columns for column in REPORT_REQUIRED_METADATA_COLUMNS),
            "processed metadata is missing report columns",
        ),
        build_check(
            "restoration_required_columns_present",
            [
                column
                for column in REPORT_REQUIRED_RESTORATION_COLUMNS
                if column not in restored_metadata_df.columns
            ],
            [],
            all(column in restored_metadata_df.columns for column in REPORT_REQUIRED_RESTORATION_COLUMNS),
            "restoration manifest is missing report columns",
        ),
        build_check(
            "restoration_expected_rows",
            int(len(restored_lama_df)),
            EXPECTED_RESTORATION_CASES,
            int(len(restored_lama_df)) == EXPECTED_RESTORATION_CASES,
            "LaMa restoration manifest row count differs from expected case universe",
        ),
        build_check(
            "restoration_ok_rows",
            int(len(restored_lama_ok_df)),
            EXPECTED_RESTORATION_CASES,
            int(len(restored_lama_ok_df)) == EXPECTED_RESTORATION_CASES,
            "LaMa restoration manifest has non-ok rows",
        ),
        build_check(
            "restoration_report_rows_excluding_zero_controls",
            int(len(report_restoration_df)),
            EXPECTED_REPORT_CASES,
            int(len(report_restoration_df)) == EXPECTED_REPORT_CASES,
            "main report case count after excluding zero controls is unexpected",
        ),
        build_check(
            "zero_control_rows_present",
            int(len(zero_control_case_ids)),
            EXPECTED_RESTORATION_CASES - EXPECTED_REPORT_CASES,
            int(len(zero_control_case_ids)) == EXPECTED_RESTORATION_CASES - EXPECTED_REPORT_CASES,
            "zero-control/empty-mask row count is unexpected",
        ),
        build_check(
            "restoration_case_ids_unique",
            int(restored_lama_df["case_id"].nunique()),
            int(len(restored_lama_df)),
            int(restored_lama_df["case_id"].nunique()) == int(len(restored_lama_df)),
            "restoration case IDs are not unique",
        ),
        build_check(
            "lpips_handoff_ready",
            lpips_handoff_manifest.get("handoff_status", ""),
            EXPECTED_REPORT_STATUS,
            lpips_handoff_manifest.get("handoff_status", "") == EXPECTED_REPORT_STATUS,
            "LPIPS handoff is not ready",
        ),
        build_check(
            "feature_handoff_ready",
            feature_handoff_manifest.get("handoff_status", ""),
            EXPECTED_REPORT_STATUS,
            feature_handoff_manifest.get("handoff_status", "") == EXPECTED_REPORT_STATUS,
            "feature-similarity handoff is not ready",
        ),
        build_check(
            "lpips_stage_batches_validated",
            lpips_stage_batch_status,
            "all true",
            all(lpips_stage_batch_status.values()),
            "one or more LPIPS stage batches is not validated",
        ),
        build_check(
            "feature_stage_batches_validated",
            feature_stage_batch_status,
            "all true",
            all(feature_stage_batch_status.values()),
            "one or more feature-similarity stage batches is not validated",
        ),
        build_check(
            "classical_report_region_available",
            classical_available_regions,
            list(CLASSICAL_REPORT_REGION_PREFERENCE),
            classical_selected_region != "",
            "classical metrics do not contain any preferred report region",
        ),
        build_check(
            "classical_report_case_coverage",
            {
                "selected_region": classical_selected_region,
                "cases": int(len(classical_report_case_ids)),
                "missing_cases": int(len(report_case_ids - classical_report_case_ids)),
                "extra_cases": int(len(classical_report_case_ids - report_case_ids)),
            },
            {
                "cases": EXPECTED_REPORT_CASES,
                "missing_cases": 0,
            },
            classical_selected_region != ""
            and len(report_case_ids - classical_report_case_ids) == 0,
            "classical selected report region does not cover all report cases",
        ),
        build_check(
            "lpips_raw_row_count",
            int(len(lpips_raw_metrics_df)),
            int(lpips_raw_expected_rows),
            int(len(lpips_raw_metrics_df)) == int(lpips_raw_expected_rows),
            "LPIPS raw row count differs from handoff contract",
        ),
        build_check(
            "lpips_raw_region_counts",
            metric_region_counts(lpips_raw_metrics_df),
            lpips_raw_expected_region_counts,
            metric_region_counts(lpips_raw_metrics_df) == lpips_raw_expected_region_counts,
            "LPIPS raw region counts differ from handoff contract",
        ),
        build_check(
            "lpips_compact_row_count_matches_raw",
            int(len(lpips_compact_metrics_df)),
            int(len(lpips_raw_metrics_df)),
            int(len(lpips_compact_metrics_df)) == int(len(lpips_raw_metrics_df)),
            "LPIPS compact metrics row count differs from raw metrics",
        ),
        build_check(
            "lpips_report_region_case_coverage",
            {
                "region": LPIPS_REPORT_REGION,
                "cases": int(len(lpips_report_case_ids)),
                "missing_cases": int(len(report_case_ids - lpips_report_case_ids)),
                "extra_cases": int(len(lpips_report_case_ids - report_case_ids)),
            },
            {
                "cases": EXPECTED_REPORT_CASES,
                "missing_cases": 0,
            },
            len(lpips_report_case_ids) == EXPECTED_REPORT_CASES
            and len(report_case_ids - lpips_report_case_ids) == 0,
            "LPIPS report region does not cover all non-zero report cases",
        ),
        build_check(
            "feature_raw_row_count",
            int(len(feature_raw_metrics_df)),
            int(feature_raw_expected_rows),
            int(len(feature_raw_metrics_df)) == int(feature_raw_expected_rows),
            "feature raw row count differs from handoff contract",
        ),
        build_check(
            "feature_raw_region_counts",
            metric_region_counts(feature_raw_metrics_df),
            feature_raw_expected_region_counts,
            metric_region_counts(feature_raw_metrics_df) == feature_raw_expected_region_counts,
            "feature raw region counts differ from handoff contract",
        ),
        build_check(
            "feature_compact_row_count_matches_raw",
            int(len(feature_compact_metrics_df)),
            int(len(feature_raw_metrics_df)),
            int(len(feature_compact_metrics_df)) == int(len(feature_raw_metrics_df)),
            "feature compact metrics row count differs from raw metrics",
        ),
        build_check(
            "feature_report_region_case_coverage",
            {
                "region": FEATURE_REPORT_REGION,
                "cases": int(len(feature_report_case_ids)),
                "missing_cases": int(len(report_case_ids - feature_report_case_ids)),
                "extra_cases": int(len(feature_report_case_ids - report_case_ids)),
            },
            {
                "cases": EXPECTED_REPORT_CASES,
                "missing_cases": 0,
            },
            len(feature_report_case_ids) == EXPECTED_REPORT_CASES
            and len(report_case_ids - feature_report_case_ids) == 0,
            "feature report region does not cover all non-zero report cases",
        ),
        build_check(
            "metric_case_region_rows_unique",
            {
                "classical": duplicate_case_region_rows(classical_metrics_df),
                "lpips_raw": duplicate_case_region_rows(lpips_raw_metrics_df),
                "lpips_compact": duplicate_case_region_rows(lpips_compact_metrics_df),
                "feature_raw": duplicate_case_region_rows(feature_raw_metrics_df),
                "feature_compact": duplicate_case_region_rows(feature_compact_metrics_df),
            },
            "all 0",
            all(
                duplicate_count == 0
                for duplicate_count in [
                    duplicate_case_region_rows(classical_metrics_df),
                    duplicate_case_region_rows(lpips_raw_metrics_df),
                    duplicate_case_region_rows(lpips_compact_metrics_df),
                    duplicate_case_region_rows(feature_raw_metrics_df),
                    duplicate_case_region_rows(feature_compact_metrics_df),
                ]
            ),
            "one or more metric tables has duplicate case/region rows",
        ),
        build_check(
            "metric_status_has_no_errors",
            {
                "classical": metric_status_counts(classical_metrics_df),
                "lpips_raw": metric_status_counts(lpips_raw_metrics_df),
                "feature_raw": metric_status_counts(feature_raw_metrics_df),
            },
            "no error statuses",
            all(
                int(dataframe["status"].astype(str).eq("error").sum()) == 0
                for dataframe in [
                    dataframe
                    for dataframe in [classical_metrics_df, lpips_raw_metrics_df, feature_raw_metrics_df]
                    if "status" in dataframe.columns
                ]
            ),
            "one or more metric tables contains error rows",
        ),
        build_check(
            "difference_map_manifest_case_and_path_columns",
            {
                "has_case_id": difference_map_case_column_present,
                "path_column": difference_map_path_column or "",
            },
            {
                "has_case_id": True,
                "path_column": "one supported path column",
            },
            difference_map_case_column_present and difference_map_path_column is not None,
            "difference-map manifest must contain case_id and a supported figure/path column",
        ),
        build_check(
            "difference_map_manifest_nonempty",
            int(len(error_map_manifest_df)),
            "> 0",
            int(len(error_map_manifest_df)) > 0,
            "difference-map manifest is empty",
        ),
    ]
)

batch2_validation_export_df = batch2_validation_df.copy()

for column in ["observed", "expected"]:
    batch2_validation_export_df[column] = batch2_validation_export_df[column].map(json_for_csv)

BATCH2_VALIDATION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
batch2_validation_export_df.to_csv(BATCH2_VALIDATION_OUTPUT_PATH, index=False)

if "stage_manifest" not in globals() or not isinstance(stage_manifest, dict):
    stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.setdefault("batches", {})
stage_manifest["batches"]["batch_2"] = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "load_inputs_and_verify_upstream_contracts",
    "validation": project_relative_path(BATCH2_VALIDATION_OUTPUT_PATH),
    "validation_passed": bool(batch2_validation_df["passed"].astype(bool).all()),
    "loaded_table_shapes": loaded_table_shapes_df.to_dict(orient="records"),
    "upstream_handoffs": upstream_handoff_df.to_dict(orient="records"),
    "contracts": {
        "expected_restoration_cases": EXPECTED_RESTORATION_CASES,
        "expected_report_cases": EXPECTED_REPORT_CASES,
        "expected_metric_region_counts": EXPECTED_METRIC_REGION_COUNTS,
        "classical_selected_region": classical_selected_region,
        "lpips_report_region": LPIPS_REPORT_REGION,
        "feature_report_region": FEATURE_REPORT_REGION,
        "zero_control_cases": int(len(zero_control_case_ids)),
        "report_case_ids": int(len(report_case_ids)),
    },
    "resolved_inputs": {
        "classical_metrics": project_relative_path(CLASSICAL_METRICS_PATH),
        "difference_map_manifest": project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH),
        "lpips_handoff_manifest": project_relative_path(LPIPS_HANDOFF_MANIFEST_PATH),
        "feature_handoff_manifest": project_relative_path(FEATURE_HANDOFF_MANIFEST_PATH),
    },
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

display(batch2_validation_df)

print("Batch 2 validation:", project_relative_path(BATCH2_VALIDATION_OUTPUT_PATH))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))
print("Classical report region:", classical_selected_region)
print("LPIPS report region:", LPIPS_REPORT_REGION)
print("Feature report region:", FEATURE_REPORT_REGION)
print("Report cases:", len(report_case_ids))
print("Zero-control cases:", len(zero_control_case_ids))

if not batch2_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Batch 2 upstream contract validation failed.")

print("Batch 2 complete.")


,check_name,observed,expected,passed,issue
0,batch1_validation_passed,True,True,True,
1,loaded_tables_nonempty,"{'processed_metadata': 50, 'restored_metadata': 410, 'classical_metrics': 2260, 'lpips_raw_metrics': 1175, 'lpips_compact_metrics': 1175, 'feature_raw_metrics': 1175, 'feature_...",all > 0,True,
2,processed_metadata_required_columns_present,[],[],True,
3,restoration_required_columns_present,[],[],True,
4,restoration_expected_rows,410,410,True,
5,restoration_ok_rows,410,410,True,
6,restoration_report_rows_excluding_zero_controls,355,355,True,
7,zero_control_rows_present,55,55,True,
8,restoration_case_ids_unique,410,410,True,
9,lpips_handoff_ready,ready_for_downstream_analysis,ready_for_downstream_analysis,True,


Batch 2 validation: outputs/19_lama_report_generation/validation/lama_report_generation_batch2_validation.csv
Stage manifest: outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json
Classical report region: masked_region
LPIPS report region: mask_bbox_crop
Feature report region: mask_bbox_crop
Report cases: 355
Zero-control cases: 55
Batch 2 complete.


In [11]:
# ============================================================
# Batch 3 - Build Report Dataframe And Compact Summaries
# ============================================================

BATCH3_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch3_validation.csv"

print("Batch 3 target:")
print("Build the main LaMa report dataframe and compact summary tables.")

Batch 3 target:
Build the main LaMa report dataframe and compact summary tables.


In [12]:
from restoration_eval.reporting import (
    prepare_lama_report_dataframe,
    summarize_metric_correlations,
    summarize_report_by_category,
    summarize_report_by_dataset,
    summarize_report_by_dataset_mask,
    summarize_report_by_mask_type,
    summarize_report_overview,
    summarize_runtime_and_failures,
)

COMPACT_SUMMARY_TABLES = [
    "overview",
    "by_dataset",
    "by_mask_type",
    "by_category",
    "by_dataset_mask",
    "runtime_and_failures",
]

In [13]:
# Avoid pandas merge suffixes for metadata fields.
# The processed metadata table is the truth source for painting-level labels.
metadata_overlap_columns = [
    "title",
    "artist",
    "date",
    "category",
    "style",
    "style_or_period",
    "medium",
    "source",
    "source_url",
    "license",
    "filename",
]

restored_metadata_report_input_df = restored_metadata_df.drop(
    columns=[
        column for column in metadata_overlap_columns
        if column in restored_metadata_df.columns
    ],
    errors="ignore",
)

In [14]:

lama_report_df, lama_report_contract = prepare_lama_report_dataframe(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=restored_metadata_report_input_df,
    classical_metrics_df=classical_metrics_df,
    lpips_metrics_df=lpips_raw_metrics_df,
    feature_metrics_df=feature_raw_metrics_df,
    error_map_manifest_df=error_map_manifest_df,
    project_root=PROJECT_ROOT,
    model_name=MODEL_NAME,
    include_zero_control=False,
)

# Re-attach figure paths from the canonical Notebook 16 manifest in the notebook
# itself. This protects Batch 4 from stale helper path-column assumptions.
difference_map_paths_df, difference_map_path_column = prepare_notebook_figure_manifest_paths(
    error_map_manifest_df,
    output_column="error_map_figure_path",
)

lama_report_df = lama_report_df.drop(columns=["error_map_figure_path"], errors="ignore").merge(
    difference_map_paths_df[["case_id", "error_map_figure_path"]].drop_duplicates("case_id"),
    on="case_id",
    how="left",
)

REPORT_DATAFRAME_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
lama_report_df.to_csv(REPORT_DATAFRAME_OUTPUT_PATH, index=False)

difference_map_paths_joined = int(
    lama_report_df["error_map_figure_path"]
    .map(lambda value: value is not None and not pd.isna(value) and str(value).strip() != "")
    .sum()
)

difference_map_paths_existing = int(
    lama_report_df["error_map_figure_path"]
    .map(lambda value: Path(str(value)).is_file() if value is not None and not pd.isna(value) else False)
    .sum()
)

# Backward-compatible alias for older validation/manifest labels.
diagnostic_paths_joined = difference_map_paths_joined

report_dataframe_shape_df = pd.DataFrame(
    [
        {"item": "report_rows", "value": int(len(lama_report_df))},
        {"item": "report_columns", "value": int(len(lama_report_df.columns))},
        {"item": "unique_case_ids", "value": int(lama_report_df["case_id"].nunique())},
        {"item": "zero_control_rows", "value": int(lama_report_df["is_zero_control"].astype(bool).sum())},
        {"item": "difference_map_paths_joined", "value": difference_map_paths_joined},
        {"item": "difference_map_paths_existing", "value": difference_map_paths_existing},
    ]
)

print("Report dataframe shape:")
display(report_dataframe_shape_df)

print("Report contract:")
display(pd.DataFrame([lama_report_contract]))

print("Report dataframe preview:")
display(
    lama_report_df[
        [
            "case_id",
            "painting_id",
            "dataset_name",
            "mask_type",
            "category",
            "mse_improvement",
            "lpips_improvement",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
            "mean_similarity_improvement",
            "error_map_figure_path",
        ]
    ].head(10)
)


Report dataframe shape:


,item,value
0,report_rows,355
1,report_columns,262
2,unique_case_ids,355
3,zero_control_rows,0
4,difference_map_paths_joined,355
5,difference_map_paths_existing,355


Report contract:


,report_schema_version,model_name,include_zero_control,classical_report_region,lpips_report_region,feature_report_region,input_rows,report_rows,report_dataset_counts,report_mask_counts
0,2.0.0,lama,False,masked_region,mask_bbox_crop,mask_bbox_crop,"{'processed_metadata': 50, 'restored_metadata': 410, 'classical_metrics': 2260, 'lpips_metrics': 1175, 'feature_metrics': 1175, 'error_map_manifest': 360}",355,"{'canonical': 200, 'mask_robustness': 75, 'synthetic_degradation': 45, 'damage_size': 35}","{nan: 155, 'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50}"


Report dataframe preview:


,case_id,painting_id,dataset_name,mask_type,category,mse_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement,mean_similarity_improvement,error_map_figure_path
0,canonical__p001_loss_large,p001,canonical,loss_large,portrait_figure,48373.386292,0.451273,0.139213,0.174818,0.157015,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_loss_large_lama_difference_maps.png
1,canonical__p001_loss_small,p001,canonical,loss_small,portrait_figure,54259.773041,0.299590,0.113468,0.056173,0.084821,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_loss_small_lama_difference_maps.png
2,canonical__p001_mixed_damage,p001,canonical,mixed_damage,portrait_figure,50582.575195,0.479921,0.286900,0.215196,0.251048,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png
3,canonical__p001_scratch_thin,p001,canonical,scratch_thin,portrait_figure,49721.899544,0.522427,0.202154,0.100681,0.151418,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png
4,canonical__p002_loss_large,p002,canonical,loss_large,portrait_figure,31630.888916,0.365739,0.274684,0.283712,0.279198,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_loss_large_lama_difference_maps.png
5,canonical__p002_loss_small,p002,canonical,loss_small,portrait_figure,43845.346581,0.280966,0.052325,0.059258,0.055791,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_loss_small_lama_difference_maps.png
6,canonical__p002_mixed_damage,p002,canonical,mixed_damage,portrait_figure,43523.208183,0.421029,0.301766,0.297415,0.299590,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_mixed_damage_lama_difference_maps.png
7,canonical__p002_scratch_thin,p002,canonical,scratch_thin,portrait_figure,48152.786285,0.364334,0.122916,0.152559,0.137738,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_scratch_thin_lama_difference_maps.png
8,canonical__p003_loss_large,p003,canonical,loss_large,portrait_figure,34918.501892,0.291022,0.205234,0.296280,0.250757,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p003_loss_large_lama_difference_maps.png
9,canonical__p003_loss_small,p003,canonical,loss_small,portrait_figure,35702.919312,0.267780,0.102540,0.048482,0.075511,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p003_loss_small_lama_difference_maps.png


In [15]:
summary_overview_df = summarize_report_overview(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=restored_metadata_df,
    report_df=lama_report_df,
)

summary_by_dataset_df = summarize_report_by_dataset(lama_report_df)
summary_by_mask_type_df = summarize_report_by_mask_type(lama_report_df)
summary_by_category_df = summarize_report_by_category(lama_report_df)
summary_by_dataset_mask_df = summarize_report_by_dataset_mask(lama_report_df)
runtime_summary_df = summarize_runtime_and_failures(restored_metadata_df)
metric_correlation_df = summarize_metric_correlations(lama_report_df)

summary_tables = {
    "overview": summary_overview_df,
    "by_dataset": summary_by_dataset_df,
    "by_mask_type": summary_by_mask_type_df,
    "by_category": summary_by_category_df,
    "by_dataset_mask": summary_by_dataset_mask_df,
    "runtime_and_failures": runtime_summary_df,
}

compact_summary_export_parts = []

for summary_name, summary_df in summary_tables.items():
    export_df = summary_df.copy()
    export_df.insert(0, "summary_table", summary_name)
    compact_summary_export_parts.append(export_df)

compact_summaries_df = pd.concat(
    compact_summary_export_parts,
    ignore_index=True,
    sort=False,
)

REPORT_SUMMARIES_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
compact_summaries_df.to_csv(REPORT_SUMMARIES_OUTPUT_PATH, index=False)
metric_correlation_df.to_csv(REPORT_CORRELATION_OUTPUT_PATH, index=True)

summary_shape_df = pd.DataFrame(
    [
        {
            "summary_table": summary_name,
            "rows": int(len(summary_df)),
            "columns": int(len(summary_df.columns)),
        }
        for summary_name, summary_df in summary_tables.items()
    ]
    + [
        {
            "summary_table": "metric_correlations",
            "rows": int(len(metric_correlation_df)),
            "columns": int(len(metric_correlation_df.columns)),
        }
    ]
)

print("Summary table shapes:")
display(summary_shape_df)

print("Overview summary:")
display(summary_overview_df)

print("Dataset summary:")
display(summary_by_dataset_df)

print("Mask-type summary:")
display(summary_by_mask_type_df)

print("Metric correlations:")
display(metric_correlation_df)

Summary table shapes:


,summary_table,rows,columns
0,overview,8,2
1,by_dataset,4,13
2,by_mask_type,5,13
3,by_category,5,13
4,by_dataset_mask,7,14
5,runtime_and_failures,4,7
6,metric_correlations,8,8


Overview summary:


,item,value
0,Paintings,50
1,Painting categories,5
2,Restoration rows,410
3,Report cases,355
4,Zero-control restoration rows,55
5,Datasets,"canonical, damage_size, mask_robustness, synthetic_degradation"
6,Mask types in report,"loss_large, loss_small, mixed_damage, scratch_thin"
7,Model,lama


Dataset summary:


,dataset_name,cases,mean_mse_improvement,median_mse_improvement,mean_lpips_improvement,median_lpips_improvement,mean_clip_similarity_improvement,median_clip_similarity_improvement,mean_dinov2_similarity_improvement,median_dinov2_similarity_improvement,mean_feature_similarity_improvement,feature_improvement_rate,mean_runtime_seconds
0,canonical,200,26852.898037,27161.071442,0.275095,0.268348,0.137435,0.129401,0.158084,0.120055,0.147759,1.0,1.460938
1,damage_size,35,26667.047511,23642.714783,0.412390,0.404265,0.212076,0.221072,0.120004,0.118968,0.166040,1.0,1.460938
2,mask_robustness,75,30922.268961,28711.128632,0.274119,0.264808,0.128423,0.116801,0.088306,0.062053,0.108365,1.0,1.460938
3,synthetic_degradation,45,-170.946195,-18.199776,-0.080465,-0.028777,-0.029774,-0.003277,-0.053592,-0.005893,-0.041683,0.0,1.460938


Mask-type summary:


,mask_type,cases,mean_mse_improvement,median_mse_improvement,mean_lpips_improvement,median_lpips_improvement,mean_clip_similarity_improvement,median_clip_similarity_improvement,mean_dinov2_similarity_improvement,median_dinov2_similarity_improvement,mean_feature_similarity_improvement,feature_improvement_rate,mean_runtime_seconds
0,loss_large,50,24443.101354,24683.065491,0.339915,0.329846,0.192740,0.191421,0.176975,0.152687,0.184857,1.000000,1.460938
1,loss_small,50,26642.677037,26131.403191,0.183860,0.183368,0.100472,0.100247,0.072543,0.049601,0.086507,1.000000,1.460938
2,mixed_damage,50,27154.329602,28490.907806,0.308045,0.316820,0.158643,0.166863,0.243539,0.185953,0.201091,1.000000,1.460938
3,scratch_thin,50,29171.484156,29626.925224,0.268560,0.270451,0.097885,0.091019,0.139279,0.109687,0.118582,1.000000,1.460938
4,NaN,155,20934.350040,22426.906769,0.202398,0.242540,0.101384,0.107541,0.054267,0.049410,0.077826,0.709677,1.460938


Metric correlations:


,mse_improvement,mae_improvement,psnr_improvement,ssim_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement,mean_similarity_improvement
mse_improvement,1.0000,0.9737,0.7561,NaN,0.6312,0.4758,0.3485,0.4443
mae_improvement,0.9737,1.0000,0.8530,NaN,0.7101,0.5518,0.4107,0.5196
psnr_improvement,0.7561,0.8530,1.0000,NaN,0.6857,0.4986,0.4002,0.4884
ssim_improvement,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lpips_improvement,0.6312,0.7101,0.6857,NaN,1.0000,0.8306,0.6277,0.7883
clip_similarity_improvement,0.4758,0.5518,0.4986,NaN,0.8306,1.0000,0.6180,0.8595
dinov2_similarity_improvement,0.3485,0.4107,0.4002,NaN,0.6277,0.6180,1.0000,0.9330
mean_similarity_improvement,0.4443,0.5196,0.4884,NaN,0.7883,0.8595,0.9330,1.0000


In [16]:
required_report_columns = [
    "case_id",
    "painting_id",
    "dataset_name",
    "mask_type",
    "model_name",
    "category",
    "mse_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
]

metric_columns_for_finite_check = [
    "mse_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
]

missing_required_report_columns = [
    column for column in required_report_columns
    if column not in lama_report_df.columns
]

finite_metric_values = True
if not missing_required_report_columns:
    finite_metric_values = bool(
        np.isfinite(
            lama_report_df[metric_columns_for_finite_check]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
        ).all()
    )

summary_tables_nonempty = {
    summary_name: int(len(summary_df))
    for summary_name, summary_df in summary_tables.items()
}

output_paths_batch3 = {
    "report_dataframe": REPORT_DATAFRAME_OUTPUT_PATH,
    "report_summaries": REPORT_SUMMARIES_OUTPUT_PATH,
    "report_metric_correlations": REPORT_CORRELATION_OUTPUT_PATH,
}

output_paths_exist = {
    name: path.is_file()
    for name, path in output_paths_batch3.items()
}

output_paths_nonempty = {
    name: int(path.stat().st_size) if path.is_file() else 0
    for name, path in output_paths_batch3.items()
}

batch3_validation_df = pd.DataFrame(
    [
        build_check(
            "batch2_validation_passed",
            validation_passed_from_csv(BATCH2_VALIDATION_OUTPUT_PATH),
            True,
            validation_passed_from_csv(BATCH2_VALIDATION_OUTPUT_PATH),
            "Batch 2 validation did not pass",
        ),
        build_check(
            "report_dataframe_expected_rows",
            int(len(lama_report_df)),
            EXPECTED_REPORT_CASES,
            int(len(lama_report_df)) == EXPECTED_REPORT_CASES,
            "report dataframe row count differs from expected non-zero case count",
        ),
        build_check(
            "report_case_ids_unique",
            int(lama_report_df["case_id"].nunique()),
            int(len(lama_report_df)),
            int(lama_report_df["case_id"].nunique()) == int(len(lama_report_df)),
            "report dataframe case IDs are not unique",
        ),
        build_check(
            "zero_controls_excluded",
            int(lama_report_df["is_zero_control"].astype(bool).sum()),
            0,
            int(lama_report_df["is_zero_control"].astype(bool).sum()) == 0,
            "zero-control rows are present in the main report dataframe",
        ),
        build_check(
            "only_lama_model_rows",
            sorted(lama_report_df["model_name"].dropna().astype(str).unique().tolist()),
            [MODEL_NAME],
            set(lama_report_df["model_name"].dropna().astype(str).unique()) == {MODEL_NAME},
            "report dataframe contains unexpected model rows",
        ),
        build_check(
            "required_report_columns_present",
            missing_required_report_columns,
            [],
            len(missing_required_report_columns) == 0,
            "report dataframe is missing required report columns",
        ),
        build_check(
            "metric_values_finite",
            finite_metric_values,
            True,
            finite_metric_values,
            "one or more report metric values is missing or non-finite",
        ),
        build_check(
            "classical_report_region_contract",
            lama_report_contract.get("classical_report_region", ""),
            classical_selected_region,
            lama_report_contract.get("classical_report_region", "") == classical_selected_region,
            "report dataframe used an unexpected classical report region",
        ),
        build_check(
            "lpips_report_region_contract",
            lama_report_contract.get("lpips_report_region", ""),
            LPIPS_REPORT_REGION,
            lama_report_contract.get("lpips_report_region", "") == LPIPS_REPORT_REGION,
            "report dataframe used an unexpected LPIPS report region",
        ),
        build_check(
            "feature_report_region_contract",
            lama_report_contract.get("feature_report_region", ""),
            FEATURE_REPORT_REGION,
            lama_report_contract.get("feature_report_region", "") == FEATURE_REPORT_REGION,
            "report dataframe used an unexpected feature report region",
        ),
        build_check(
            "compact_summary_tables_nonempty",
            summary_tables_nonempty,
            "all > 0",
            all(value > 0 for value in summary_tables_nonempty.values()),
            "one or more compact summary tables is empty",
        ),
        build_check(
            "metric_correlation_table_nonempty",
            int(len(metric_correlation_df)),
            "> 0",
            int(len(metric_correlation_df)) > 0,
            "metric correlation table is empty",
        ),
        build_check(
            "batch3_outputs_exist",
            output_paths_exist,
            "all true",
            all(output_paths_exist.values()),
            "one or more Batch 3 output artifacts is missing",
        ),
        build_check(
            "batch3_outputs_nonempty",
            output_paths_nonempty,
            "all > 0",
            all(value > 0 for value in output_paths_nonempty.values()),
            "one or more Batch 3 output artifacts is empty",
        ),
        build_check(
            "difference_map_paths_joined",
            difference_map_paths_joined,
            EXPECTED_REPORT_CASES,
            difference_map_paths_joined >= EXPECTED_REPORT_CASES,
            "difference-map manifest did not cover all report cases",
        ),
    ]
)

batch3_validation_export_df = batch3_validation_df.copy()

for column in ["observed", "expected"]:
    batch3_validation_export_df[column] = batch3_validation_export_df[column].map(json_for_csv)

BATCH3_VALIDATION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
batch3_validation_export_df.to_csv(BATCH3_VALIDATION_OUTPUT_PATH, index=False)

if "stage_manifest" not in globals() or not isinstance(stage_manifest, dict):
    stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.setdefault("batches", {})
stage_manifest["batches"]["batch_3"] = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "build_report_dataframe_and_compact_summaries",
    "validation": project_relative_path(BATCH3_VALIDATION_OUTPUT_PATH),
    "validation_passed": bool(batch3_validation_df["passed"].astype(bool).all()),
    "outputs": {
        name: project_relative_path(path)
        for name, path in output_paths_batch3.items()
    },
    "report_contract": lama_report_contract,
    "table_shapes": {
        "report_dataframe": {
            "rows": int(len(lama_report_df)),
            "columns": int(len(lama_report_df.columns)),
        },
        "compact_summaries": {
            "rows": int(len(compact_summaries_df)),
            "columns": int(len(compact_summaries_df.columns)),
        },
        "metric_correlations": {
            "rows": int(len(metric_correlation_df)),
            "columns": int(len(metric_correlation_df.columns)),
        },
    },
    "summary_table_shapes": summary_shape_df.to_dict(orient="records"),
    "difference_map_paths_joined": difference_map_paths_joined,
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

display(batch3_validation_df)

print("Report dataframe:", project_relative_path(REPORT_DATAFRAME_OUTPUT_PATH))
print("Compact summaries:", project_relative_path(REPORT_SUMMARIES_OUTPUT_PATH))
print("Metric correlations:", project_relative_path(REPORT_CORRELATION_OUTPUT_PATH))
print("Batch 3 validation:", project_relative_path(BATCH3_VALIDATION_OUTPUT_PATH))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

if not batch3_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Batch 3 report dataframe and summary validation failed.")

print("Batch 3 complete.")


,check_name,observed,expected,passed,issue
0,batch2_validation_passed,True,True,True,
1,report_dataframe_expected_rows,355,355,True,
2,report_case_ids_unique,355,355,True,
3,zero_controls_excluded,0,0,True,
4,only_lama_model_rows,[lama],[lama],True,
5,required_report_columns_present,[],[],True,
6,metric_values_finite,True,True,True,
7,classical_report_region_contract,masked_region,masked_region,True,
8,lpips_report_region_contract,mask_bbox_crop,mask_bbox_crop,True,
9,feature_report_region_contract,mask_bbox_crop,mask_bbox_crop,True,


Report dataframe: outputs/19_lama_report_generation/metrics/lama_report_dataframe.csv
Compact summaries: outputs/19_lama_report_generation/metrics/lama_report_summaries.csv
Metric correlations: outputs/19_lama_report_generation/metrics/lama_report_metric_correlations.csv
Batch 3 validation: outputs/19_lama_report_generation/validation/lama_report_generation_batch3_validation.csv
Stage manifest: outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json
Batch 3 complete.


In [17]:

# ============================================================
# Batch 4 - Select Diagnostic Cases And Validate Figure References
# ============================================================

from restoration_eval.reporting import (
    select_lama_diagnostic_cases,
)

BATCH4_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch4_validation.csv"

MIN_SELECTED_DIAGNOSTIC_CASES = 8
MAX_SELECTED_DIAGNOSTIC_CASES = 18
TARGET_DIFFERENCE_MAP_CASES = min(12, MAX_SELECTED_DIAGNOSTIC_CASES)

print("Batch 4 target:")
print("Select representative cases backed by the Notebook 16 difference-map manifest.")


Batch 4 target:
Select representative cases backed by the Notebook 16 difference-map manifest.


In [18]:

# Batch 4 uses local manifest-path helpers defined above, so it does not need a
# helper reload here.
print("Using canonical difference-map manifest:", project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH))
print("Notebook 19 output root:", project_relative_path(NOTEBOOK_OUTPUT_DIR))


Using canonical difference-map manifest: outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv
Notebook 19 output root: outputs/19_lama_report_generation


In [19]:

difference_map_paths_df, difference_map_path_column = prepare_notebook_figure_manifest_paths(
    error_map_manifest_df,
    output_column="error_map_figure_path",
)

difference_map_paths_df["error_map_figure_path_exists"] = difference_map_paths_df[
    "error_map_figure_path"
].map(lambda value: Path(str(value)).is_file() if value is not None and not pd.isna(value) else False)

difference_map_paths_df["error_map_figure_path_project_relative"] = difference_map_paths_df[
    "error_map_figure_path"
].map(
    lambda value: project_relative_path(Path(str(value)))
    if value is not None and not pd.isna(value) and str(value).strip()
    else ""
)

report_case_ids_for_figures = set(lama_report_df["case_id"].astype(str))
figure_manifest_case_ids = set(difference_map_paths_df["case_id"].astype(str))
figure_report_overlap_ids = report_case_ids_for_figures & figure_manifest_case_ids

optional_diagnostic_panel_df = pd.DataFrame()
optional_diagnostic_panel_count = 0
if OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH.is_file():
    optional_diagnostic_panel_df = pd.read_csv(OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH, low_memory=False)
    optional_diagnostic_panel_df = normalise_case_id_for_contract(optional_diagnostic_panel_df)
    optional_diagnostic_panel_count = int(len(optional_diagnostic_panel_df))

# Replace/refresh the report dataframe path column after path resolution.
lama_report_df = lama_report_df.drop(
    columns=["error_map_figure_path", "error_map_figure_path_exists", "error_map_figure_path_project_relative"],
    errors="ignore",
).merge(
    difference_map_paths_df[
        [
            "case_id",
            "error_map_figure_path",
            "error_map_figure_path_exists",
            "error_map_figure_path_project_relative",
        ]
    ],
    on="case_id",
    how="left",
)

lama_report_df.to_csv(REPORT_DATAFRAME_OUTPUT_PATH, index=False)

# Backward-compatible alias so remaining report-helper naming still works.
diagnostic_panel_paths_df = difference_map_paths_df
diagnostic_path_column = difference_map_path_column

print("Difference-map manifest reference summary:")
display(
    pd.DataFrame(
        [
            {
                "manifest_path": project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH),
                "rows": int(len(error_map_manifest_df)),
                "path_column": difference_map_path_column or "",
                "unique_case_ids": int(error_map_manifest_df["case_id"].nunique())
                if "case_id" in error_map_manifest_df.columns
                else 0,
                "report_overlap_case_ids": int(len(figure_report_overlap_ids)),
                "resolved_figure_rows": int(len(difference_map_paths_df)),
                "existing_figure_files": int(difference_map_paths_df["error_map_figure_path_exists"].sum()),
                "optional_diagnostic_panel_rows": optional_diagnostic_panel_count,
            }
        ]
    )
)

display(
    difference_map_paths_df[
        [
            column
            for column in [
                "case_id",
                "painting_id",
                "dataset_name",
                "mask_type",
                "error_map_figure_path",
                "error_map_figure_path_project_relative",
                "error_map_figure_path_exists",
                "status",
                "figure_type",
            ]
            if column in difference_map_paths_df.columns
        ]
    ].head(20)
)


Difference-map manifest reference summary:


,manifest_path,rows,path_column,unique_case_ids,report_overlap_case_ids,resolved_figure_rows,existing_figure_files,optional_diagnostic_panel_rows
0,outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv,360,figure_path,360,355,360,360,12


,case_id,painting_id,dataset_name,mask_type,error_map_figure_path,error_map_figure_path_project_relative,error_map_figure_path_exists,status
0,canonical__p001_loss_large,p001,canonical,loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_loss_large_lama_difference_maps.png,True,ok
1,canonical__p001_loss_small,p001,canonical,loss_small,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_loss_small_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_loss_small_lama_difference_maps.png,True,ok
2,canonical__p001_mixed_damage,p001,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png,True,ok
3,canonical__p001_scratch_thin,p001,canonical,scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png,True,ok
4,canonical__p002_loss_large,p002,canonical,loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_loss_large_lama_difference_maps.png,True,ok
5,canonical__p002_loss_small,p002,canonical,loss_small,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_loss_small_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_loss_small_lama_difference_maps.png,True,ok
6,canonical__p002_mixed_damage,p002,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_mixed_damage_lama_difference_maps.png,True,ok
7,canonical__p002_scratch_thin,p002,canonical,scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_scratch_thin_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_scratch_thin_lama_difference_maps.png,True,ok
8,canonical__p003_loss_large,p003,canonical,loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p003_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p003_loss_large_lama_difference_maps.png,True,ok
9,canonical__p003_loss_small,p003,canonical,loss_small,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p003_loss_small_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p003_loss_small_lama_difference_maps.png,True,ok


In [20]:

difference_map_overlap_debug_df = difference_map_paths_df.loc[
    difference_map_paths_df["case_id"].astype(str).isin(
        set(lama_report_df["case_id"].astype(str))
    )
].copy()

debug_columns = [
    column
    for column in [
        "case_id",
        "error_map_figure_path",
        "error_map_figure_path_project_relative",
        "error_map_figure_path_exists",
        "dataset_name",
        "mask_type",
        "status",
    ]
    if column in difference_map_overlap_debug_df.columns
]

print("Difference-map/report overlap rows:", len(difference_map_overlap_debug_df))
print(
    "Existing overlap difference-map files:",
    int(difference_map_overlap_debug_df["error_map_figure_path_exists"].sum())
)

display(difference_map_overlap_debug_df[debug_columns].head(30))


Difference-map/report overlap rows: 355
Existing overlap difference-map files: 355


,case_id,error_map_figure_path,error_map_figure_path_project_relative,error_map_figure_path_exists,dataset_name,mask_type,status
0,canonical__p001_loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_loss_large_lama_difference_maps.png,True,canonical,loss_large,ok
1,canonical__p001_loss_small,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_loss_small_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_loss_small_lama_difference_maps.png,True,canonical,loss_small,ok
2,canonical__p001_mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png,True,canonical,mixed_damage,ok
3,canonical__p001_scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png,True,canonical,scratch_thin,ok
4,canonical__p002_loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_loss_large_lama_difference_maps.png,True,canonical,loss_large,ok
5,canonical__p002_loss_small,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_loss_small_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_loss_small_lama_difference_maps.png,True,canonical,loss_small,ok
6,canonical__p002_mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_mixed_damage_lama_difference_maps.png,True,canonical,mixed_damage,ok
7,canonical__p002_scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p002_scratch_thin_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p002_scratch_thin_lama_difference_maps.png,True,canonical,scratch_thin,ok
8,canonical__p003_loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p003_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p003_loss_large_lama_difference_maps.png,True,canonical,loss_large,ok
9,canonical__p003_loss_small,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p003_loss_small_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p003_loss_small_lama_difference_maps.png,True,canonical,loss_small,ok


In [21]:

# ============================================================
# Cell 21 - Select Representative Cases With Difference-Map Quota
# ============================================================

metric_selected_candidates_df = select_lama_diagnostic_cases(
    lama_report_df,
    n_per_signal=4,
    include_dataset_examples=True,
    include_category_examples=True,
)

metric_selection_reason_df = (
    metric_selected_candidates_df[
        ["case_id", "selection_reason", "selection_metric"]
    ]
    .drop_duplicates("case_id")
    .copy()
)

metric_selection_reason_df["case_id"] = metric_selection_reason_df["case_id"].astype(str)

report_case_ids = set(lama_report_df["case_id"].astype(str))
figure_case_ids = set(difference_map_paths_df["case_id"].astype(str))
metric_candidate_ids = set(metric_selected_candidates_df["case_id"].astype(str))

print("Difference-map manifest cases:", len(figure_case_ids))
print("Report dataframe cases:", len(report_case_ids))
print("Difference-map/report overlap:", len(figure_case_ids & report_case_ids))
print("Metric-selected candidate cases:", len(metric_candidate_ids))
print("Metric/difference-map overlap:", len(metric_candidate_ids & figure_case_ids))

source_image_columns = [
    column
    for column in [
        "clean_path_resolved",
        "damaged_path_resolved",
        "restored_path_resolved",
        "mask_path_resolved",
    ]
    if column in lama_report_df.columns
]

report_source_backed_df = lama_report_df.copy()
report_source_backed_df["case_id"] = report_source_backed_df["case_id"].astype(str)

for column in source_image_columns:
    report_source_backed_df[f"{column}_exists"] = report_source_backed_df[column].map(
        lambda value: Path(str(value)).is_file()
        if value is not None and not pd.isna(value) and str(value).strip()
        else False
    )

report_source_backed_df["source_images_available"] = (
    report_source_backed_df[[f"{column}_exists" for column in source_image_columns]].all(axis=1)
    if source_image_columns
    else False
)

report_source_backed_df["error_map_figure_path_exists"] = report_source_backed_df[
    "error_map_figure_path_exists"
].fillna(False).astype(bool)

report_source_backed_df = report_source_backed_df.loc[
    report_source_backed_df["source_images_available"].astype(bool)
].copy()

report_source_backed_df = report_source_backed_df.merge(
    metric_selection_reason_df,
    on="case_id",
    how="left",
)

report_source_backed_df["metric_candidate"] = report_source_backed_df["case_id"].isin(
    metric_candidate_ids
)

priority_metric_columns = [
    column
    for column in [
        "mse_improvement",
        "lpips_improvement",
        "clip_similarity_improvement",
        "dinov2_similarity_improvement",
        "mean_similarity_improvement",
    ]
    if column in report_source_backed_df.columns
]

priority_score = pd.Series(0.0, index=report_source_backed_df.index)

for column in priority_metric_columns:
    numeric_values = pd.to_numeric(report_source_backed_df[column], errors="coerce").abs()
    priority_score = priority_score + numeric_values.rank(pct=True).fillna(0.0)

report_source_backed_df["selection_priority_score"] = priority_score


def combined_selection_reason(row: pd.Series) -> str:
    reasons = []

    metric_reason = row.get("selection_reason", "")
    if metric_reason is not None and not pd.isna(metric_reason) and str(metric_reason).strip():
        reasons.append(str(metric_reason).strip())

    if row.get("error_map_figure_path_exists", False):
        reasons.append("Notebook 16 difference-map figure available")

    if not reasons:
        reasons.append("Representative source-image-backed report case")

    return " | ".join(reasons)


report_source_backed_df["selection_reason"] = report_source_backed_df.apply(
    combined_selection_reason,
    axis=1,
)

report_source_backed_df["selection_metric"] = report_source_backed_df[
    "selection_metric"
].fillna("source_image_available")

figure_backed_pool_df = (
    report_source_backed_df.loc[
        report_source_backed_df["error_map_figure_path_exists"].astype(bool)
    ]
    .sort_values(
        ["metric_candidate", "selection_priority_score", "dataset_name", "mask_type", "case_id"],
        ascending=[False, False, True, True, True],
        kind="stable",
    )
)

TARGET_DIFFERENCE_MAP_CASES = min(
    12,
    MAX_SELECTED_DIAGNOSTIC_CASES,
    int(len(figure_backed_pool_df)),
)

figure_backed_selected_df = figure_backed_pool_df.head(TARGET_DIFFERENCE_MAP_CASES)

metric_selected_df = (
    report_source_backed_df.loc[report_source_backed_df["metric_candidate"].astype(bool)]
    .sort_values(
        ["error_map_figure_path_exists", "selection_priority_score", "dataset_name", "mask_type", "case_id"],
        ascending=[False, False, True, True, True],
        kind="stable",
    )
)

dataset_examples_df = (
    report_source_backed_df.sort_values("selection_priority_score", ascending=False, kind="stable")
    .groupby("dataset_name", dropna=False)
    .head(1)
    if "dataset_name" in report_source_backed_df.columns
    else report_source_backed_df.head(0)
)

mask_examples_df = (
    report_source_backed_df.sort_values("selection_priority_score", ascending=False, kind="stable")
    .groupby("mask_type", dropna=False)
    .head(1)
    if "mask_type" in report_source_backed_df.columns
    else report_source_backed_df.head(0)
)

category_examples_df = (
    report_source_backed_df.sort_values("selection_priority_score", ascending=False, kind="stable")
    .groupby("category", dropna=False)
    .head(1)
    if "category" in report_source_backed_df.columns
    else report_source_backed_df.head(0)
)

fallback_df = report_source_backed_df.sort_values(
    ["error_map_figure_path_exists", "metric_candidate", "selection_priority_score", "dataset_name", "mask_type", "case_id"],
    ascending=[False, False, False, True, True, True],
    kind="stable",
)

selected_cases_df = (
    pd.concat(
        [
            figure_backed_selected_df,
            metric_selected_df,
            dataset_examples_df,
            mask_examples_df,
            category_examples_df,
            fallback_df,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates("case_id")
    .head(MAX_SELECTED_DIAGNOSTIC_CASES)
    .reset_index(drop=True)
)

# Keep this alias so existing Batch 4 manifest code still works.
report_image_backed_df = report_source_backed_df

SELECTED_CASES_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
selected_cases_df.to_csv(SELECTED_CASES_OUTPUT_PATH, index=False)

selected_case_display_columns = [
    column
    for column in [
        "case_id",
        "painting_id",
        "title",
        "dataset_name",
        "category",
        "mask_type",
        "selection_reason",
        "mse_improvement",
        "lpips_improvement",
        "mean_similarity_improvement",
        "metric_candidate",
        "error_map_figure_path_exists",
        "error_map_figure_path_project_relative",
    ]
    if column in selected_cases_df.columns
]

print("Source-image-backed report cases available:", len(report_source_backed_df))
print("Difference-map-backed report cases available:", len(figure_backed_pool_df))
print("Target difference-map cases:", TARGET_DIFFERENCE_MAP_CASES)
print("Selected cases for report:", len(selected_cases_df))
print(
    "Selected cases with difference-map figures:",
    int(selected_cases_df["error_map_figure_path_exists"].sum()),
)

display(selected_cases_df[selected_case_display_columns])


Difference-map manifest cases: 360
Report dataframe cases: 355
Difference-map/report overlap: 355
Metric-selected candidate cases: 25
Metric/difference-map overlap: 25
Source-image-backed report cases available: 355
Difference-map-backed report cases available: 355
Target difference-map cases: 12
Selected cases for report: 18
Selected cases with difference-map figures: 18


,case_id,painting_id,title,dataset_name,category,mask_type,selection_reason,mse_improvement,lpips_improvement,mean_similarity_improvement,metric_candidate,error_map_figure_path_exists,error_map_figure_path_project_relative
0,mask_robustness__p001__loss_large__variant_03,p001,Juan de Pareja,mask_robustness,portrait_figure,NaN,Representative high-scoring category example; Representative high-scoring dataset example | Notebook 16 difference-map figure available,48786.026550,0.516388,0.351512,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_mask_robustness_mask_robustness_p001_loss_large_variant_03_lama_difference_maps.png
1,canonical__p050_mixed_damage,p050,Landscape with a Sunlit Stream,canonical,high_texture_brushwork,mixed_damage,Representative high-scoring category example; Representative high-scoring dataset example; Strongest local feature-similarity improvement | Notebook 16 difference-map figure av...,40351.888306,0.422922,0.457823,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p050_mixed_damage_lama_difference_maps.png
2,canonical__p001_mixed_damage,p001,Juan de Pareja,canonical,portrait_figure,mixed_damage,Slowest restoration runtime | Notebook 16 difference-map figure available,50582.575195,0.479921,0.251048,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png
3,canonical__p027_loss_large,p027,Italian Landscape with the Ponte Lucano over the Aniene River,canonical,architecture_structured,loss_large,Representative high-scoring category example | Notebook 16 difference-map figure available,29947.750305,0.515375,0.325921,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p027_loss_large_lama_difference_maps.png
4,canonical__p006_scratch_thin,p006,Boy with a Sword,canonical,portrait_figure,scratch_thin,Strongest local MSE improvement | Notebook 16 difference-map figure available,53519.212227,0.415661,0.225047,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p006_scratch_thin_lama_difference_maps.png
5,canonical__p013_mixed_damage,p013,Landscape with a Village in the Distance,canonical,landscape_natural,mixed_damage,Representative high-scoring category example; Strongest local feature-similarity improvement | Notebook 16 difference-map figure available,38569.472046,0.370276,0.372580,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p013_mixed_damage_lama_difference_maps.png
6,canonical__p044_mixed_damage,p044,Versailles,canonical,high_texture_brushwork,mixed_damage,Strongest local feature-similarity improvement | Notebook 16 difference-map figure available,28698.513367,0.353440,0.391332,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p044_mixed_damage_lama_difference_maps.png
7,canonical__p011_loss_large,p011,View of Haarlem and the Haarlemmer Meer,canonical,landscape_natural,loss_large,Strongest local LPIPS improvement | Notebook 16 difference-map figure available,18223.183861,0.560803,0.280961,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p011_loss_large_lama_difference_maps.png
8,canonical__p014_mixed_damage,p014,Landscape on a River,canonical,landscape_natural,mixed_damage,Strongest local feature-similarity improvement | Notebook 16 difference-map figure available,29267.156128,0.312184,0.355790,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p014_mixed_damage_lama_difference_maps.png
9,canonical__p001_scratch_thin,p001,Juan de Pareja,canonical,portrait_figure,scratch_thin,Slowest restoration runtime | Notebook 16 difference-map figure available,49721.899544,0.522427,0.151418,True,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_scratch_thin_lama_

In [22]:

source_path_columns = [
    column
    for column in [
        "clean_path_resolved",
        "damaged_path_resolved",
        "restored_path_resolved",
        "mask_path_resolved",
    ]
    if column in selected_cases_df.columns
]

selected_figure_reference_df = selected_cases_df[
    [
        column
        for column in [
            "case_id",
            "painting_id",
            "dataset_name",
            "mask_type",
            "error_map_figure_path",
            "error_map_figure_path_project_relative",
        ]
        if column in selected_cases_df.columns
    ]
].copy()

selected_figure_reference_df["error_map_exists"] = selected_figure_reference_df[
    "error_map_figure_path"
].map(lambda value: Path(str(value)).is_file() if value is not None and not pd.isna(value) else False)

selected_figure_reference_df["error_map_size_mb"] = selected_figure_reference_df[
    "error_map_figure_path"
].map(
    lambda value: round(Path(str(value)).stat().st_size / (1024 * 1024), 3)
    if value is not None and not pd.isna(value) and Path(str(value)).is_file()
    else 0.0
)

source_path_validation_records = []

for _, row in selected_cases_df.iterrows():
    for column in source_path_columns:
        value = row.get(column, None)
        path = Path(str(value)) if value is not None and not pd.isna(value) else None
        source_path_validation_records.append(
            {
                "case_id": row.get("case_id", ""),
                "path_role": column,
                "path": project_relative_path(path) if path is not None else "",
                "exists": bool(path is not None and path.is_file()),
            }
        )

source_path_validation_df = pd.DataFrame(source_path_validation_records)

selected_dataset_count = (
    int(selected_cases_df["dataset_name"].nunique())
    if "dataset_name" in selected_cases_df.columns
    else 0
)
report_dataset_count = (
    int(lama_report_df["dataset_name"].nunique())
    if "dataset_name" in lama_report_df.columns
    else 0
)

selected_mask_count = (
    int(selected_cases_df["mask_type"].nunique())
    if "mask_type" in selected_cases_df.columns
    else 0
)
report_mask_count = (
    int(lama_report_df["mask_type"].nunique())
    if "mask_type" in lama_report_df.columns
    else 0
)

selected_category_count = (
    int(selected_cases_df["category"].nunique())
    if "category" in selected_cases_df.columns
    else 0
)
report_category_count = (
    int(lama_report_df["category"].nunique())
    if "category" in lama_report_df.columns
    else 0
)

print("Selected difference-map figure references:")
display(selected_figure_reference_df)

print("Selected source-image reference checks:")
display(source_path_validation_df)


Selected difference-map figure references:


,case_id,painting_id,dataset_name,mask_type,error_map_figure_path,error_map_figure_path_project_relative,error_map_exists,error_map_size_mb
0,mask_robustness__p001__loss_large__variant_03,p001,mask_robustness,NaN,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_mask_robustness_mask_robustness_p001_loss_large_variant_03_lama_...,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_mask_robustness_mask_robustness_p001_loss_large_variant_03_lama_difference_maps.png,True,2.158
1,canonical__p050_mixed_damage,p050,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p050_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p050_mixed_damage_lama_difference_maps.png,True,2.522
2,canonical__p001_mixed_damage,p001,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png,True,2.297
3,canonical__p027_loss_large,p027,canonical,loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p027_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p027_loss_large_lama_difference_maps.png,True,2.437
4,canonical__p006_scratch_thin,p006,canonical,scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p006_scratch_thin_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p006_scratch_thin_lama_difference_maps.png,True,1.496
5,canonical__p013_mixed_damage,p013,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p013_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p013_mixed_damage_lama_difference_maps.png,True,2.223
6,canonical__p044_mixed_damage,p044,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p044_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p044_mixed_damage_lama_difference_maps.png,True,2.392
7,canonical__p011_loss_large,p011,canonical,loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p011_loss_large_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p011_loss_large_lama_difference_maps.png,True,1.816
8,canonical__p014_mixed_damage,p014,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p014_mixed_damage_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p014_mixed_damage_lama_difference_maps.png,True,1.662
9,canonical__p001_scratch_thin,p001,canonical,scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases\all_cases\lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_scratch_thin_lama_difference_maps.png,True,2.191


Selected source-image reference checks:


,case_id,path_role,path,exists
0,mask_robustness__p001__loss_large__variant_03,clean_path_resolved,data/processed/clean/p001_clean.png,True
1,mask_robustness__p001__loss_large__variant_03,damaged_path_resolved,data/processed/damaged/mask_robustness/p001__loss_large__variant_03_damaged.png,True
2,mask_robustness__p001__loss_large__variant_03,restored_path_resolved,data/processed/restored/lama/mask_robustness/mask_robustness__p001__loss_large__variant_03_restored_lama.png,True
3,mask_robustness__p001__loss_large__variant_03,mask_path_resolved,data/processed/masks/mask_robustness/p001__loss_large__variant_03_mask.png,True
4,canonical__p050_mixed_damage,clean_path_resolved,data/processed/clean/p050_clean.png,True
...,...,...,...,...
67,damage_size__p026__loss_large__size_02pct,mask_path_resolved,data/processed/masks/damage_size_sensitivity/p026__loss_large__size_02pct_mask.png,True
68,synthetic_degradation__p043__partial_transparency__severe,clean_path_resolved,data/processed/clean/p043_clean.png,True
69,synthetic_degradation__p043__partial_transparency__severe,damaged_path_resolved,data/processed/degraded/synthetic_degradation/p043__partial_transparency__severe_degraded.png,True
70,synthetic_degradation__p043__partial_transparency__severe,restored_path_resolved,data/processed/restored/lama/synthetic_degradation/synthetic_degradation__p043__partial_transparency__severe_restored_lama.png,True


In [23]:

selected_case_ids = set(selected_cases_df["case_id"].astype(str))
report_case_ids_from_df = set(lama_report_df["case_id"].astype(str))

selected_case_output_exists = SELECTED_CASES_OUTPUT_PATH.is_file()
selected_case_output_size = (
    int(SELECTED_CASES_OUTPUT_PATH.stat().st_size)
    if selected_case_output_exists
    else 0
)

all_selected_error_maps_exist = (
    bool(selected_figure_reference_df["error_map_exists"].all())
    if not selected_figure_reference_df.empty
    else False
)

all_selected_source_paths_exist = (
    bool(source_path_validation_df["exists"].all())
    if not source_path_validation_df.empty
    else True
)

figure_manifest_report_overlap_count = int(len(figure_report_overlap_ids))
existing_difference_map_files = int(difference_map_paths_df["error_map_figure_path_exists"].sum())
selected_existing_difference_map_files = int(selected_figure_reference_df["error_map_exists"].sum())

batch4_validation_df = pd.DataFrame(
    [
        build_check(
            "batch3_validation_passed",
            validation_passed_from_csv(BATCH3_VALIDATION_OUTPUT_PATH),
            True,
            validation_passed_from_csv(BATCH3_VALIDATION_OUTPUT_PATH),
            "Batch 3 validation did not pass",
        ),
        build_check(
            "difference_map_manifest_path_column_present",
            difference_map_path_column or "",
            "one supported difference-map path column",
            difference_map_path_column is not None,
            "difference-map manifest does not contain a supported image path column",
        ),
        build_check(
            "difference_map_manifest_report_overlap",
            figure_manifest_report_overlap_count,
            f">= {MIN_SELECTED_DIAGNOSTIC_CASES}",
            figure_manifest_report_overlap_count >= MIN_SELECTED_DIAGNOSTIC_CASES,
            "too few report cases overlap with the difference-map manifest",
        ),
        build_check(
            "difference_map_files_available",
            existing_difference_map_files,
            f">= {MIN_SELECTED_DIAGNOSTIC_CASES}",
            existing_difference_map_files >= MIN_SELECTED_DIAGNOSTIC_CASES,
            "too few difference-map files exist for selected case sections",
        ),
        build_check(
            "selected_cases_nonempty",
            int(len(selected_cases_df)),
            "> 0",
            int(len(selected_cases_df)) > 0,
            "selected cases dataframe is empty",
        ),
        build_check(
            "selected_cases_size",
            int(len(selected_cases_df)),
            f"{MIN_SELECTED_DIAGNOSTIC_CASES} to {MAX_SELECTED_DIAGNOSTIC_CASES}",
            MIN_SELECTED_DIAGNOSTIC_CASES <= int(len(selected_cases_df)) <= MAX_SELECTED_DIAGNOSTIC_CASES,
            "selected case count is outside the planned compact report range",
        ),
        build_check(
            "selected_case_ids_unique",
            int(selected_cases_df["case_id"].nunique()),
            int(len(selected_cases_df)),
            int(selected_cases_df["case_id"].nunique()) == int(len(selected_cases_df)),
            "selected cases contain duplicate case IDs",
        ),
        build_check(
            "selected_cases_subset_of_report",
            {
                "selected_cases": int(len(selected_case_ids)),
                "not_in_report": sorted(selected_case_ids - report_case_ids_from_df)[:10],
            },
            {"not_in_report": []},
            len(selected_case_ids - report_case_ids_from_df) == 0,
            "one or more selected cases is not present in the main report dataframe",
        ),
        build_check(
            "selected_difference_map_paths_exist",
            selected_existing_difference_map_files,
            int(len(selected_figure_reference_df)),
            all_selected_error_maps_exist,
            "one or more selected difference-map figure paths is missing",
        ),
        build_check(
            "selected_source_image_paths_exist",
            {
                "checked_paths": int(len(source_path_validation_df)),
                "missing_paths": int((~source_path_validation_df["exists"]).sum())
                if not source_path_validation_df.empty
                else 0,
            },
            {"missing_paths": 0},
            all_selected_source_paths_exist,
            "one or more clean/damaged/restored/mask source image paths is missing",
        ),
        build_check(
            "selected_dataset_coverage",
            selected_dataset_count,
            f">= min(2, report datasets={report_dataset_count})",
            selected_dataset_count >= min(2, report_dataset_count),
            "selected cases do not cover enough dataset groups",
        ),
        build_check(
            "selected_mask_coverage",
            selected_mask_count,
            f">= min(2, report masks={report_mask_count})",
            selected_mask_count >= min(2, report_mask_count),
            "selected cases do not cover enough mask groups",
        ),
        build_check(
            "selected_category_coverage",
            selected_category_count,
            f">= min(2, report categories={report_category_count})",
            selected_category_count >= min(2, report_category_count),
            "selected cases do not cover enough painting categories",
        ),
        build_check(
            "selection_reasons_present",
            int(
                selected_cases_df["selection_reason"]
                .fillna("")
                .astype(str)
                .str.strip()
                .ne("")
                .sum()
            ),
            int(len(selected_cases_df)),
            int(
                selected_cases_df["selection_reason"]
                .fillna("")
                .astype(str)
                .str.strip()
                .ne("")
                .sum()
            )
            == int(len(selected_cases_df)),
            "one or more selected cases lacks a selection reason",
        ),
        build_check(
            "selected_cases_output_exists",
            selected_case_output_exists,
            True,
            selected_case_output_exists,
            "selected cases CSV was not written",
        ),
        build_check(
            "selected_cases_output_nonempty",
            selected_case_output_size,
            "> 0",
            selected_case_output_size > 0,
            "selected cases CSV is empty",
        ),
    ]
)

batch4_validation_export_df = batch4_validation_df.copy()

for column in ["observed", "expected"]:
    batch4_validation_export_df[column] = batch4_validation_export_df[column].map(json_for_csv)

BATCH4_VALIDATION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
batch4_validation_export_df.to_csv(BATCH4_VALIDATION_OUTPUT_PATH, index=False)

if "stage_manifest" not in globals() or not isinstance(stage_manifest, dict):
    stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.setdefault("batches", {})
stage_manifest["batches"]["batch_4"] = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "select_diagnostic_cases_and_validate_difference_map_figures",
    "validation": project_relative_path(BATCH4_VALIDATION_OUTPUT_PATH),
    "validation_passed": bool(batch4_validation_df["passed"].astype(bool).all()),
    "outputs": {
        "selected_cases": project_relative_path(SELECTED_CASES_OUTPUT_PATH),
        "report_dataframe_refreshed": project_relative_path(REPORT_DATAFRAME_OUTPUT_PATH),
    },
    "selection_policy": {
        "source_image_backed_cases_only": True,
        "difference_map_backed_cases_preferred": True,
        "min_selected_cases": MIN_SELECTED_DIAGNOSTIC_CASES,
        "max_selected_cases": MAX_SELECTED_DIAGNOSTIC_CASES,
        "target_difference_map_cases": TARGET_DIFFERENCE_MAP_CASES,
        "metric_candidate_cases": int(len(metric_selected_candidates_df)),
        "source_image_backed_available_cases": int(len(report_image_backed_df)),
        "difference_map_backed_available_cases": int(len(figure_backed_pool_df)),
        "selected_cases": int(len(selected_cases_df)),
    },
    "coverage": {
        "datasets": selected_dataset_count,
        "mask_types": selected_mask_count,
        "categories": selected_category_count,
    },
    "figure_reference_summary": {
        "difference_map_manifest": project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH),
        "difference_map_manifest_rows": int(len(error_map_manifest_df)),
        "difference_map_path_column": difference_map_path_column or "",
        "report_overlap_case_ids": figure_manifest_report_overlap_count,
        "existing_difference_map_files": existing_difference_map_files,
        "selected_difference_map_files": selected_existing_difference_map_files,
        "optional_diagnostic_panel_manifest": project_relative_path(OPTIONAL_DIAGNOSTIC_PANEL_MANIFEST_PATH),
        "optional_diagnostic_panel_rows": optional_diagnostic_panel_count,
    },
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

display(batch4_validation_df)

print("Selected cases:", project_relative_path(SELECTED_CASES_OUTPUT_PATH))
print("Batch 4 validation:", project_relative_path(BATCH4_VALIDATION_OUTPUT_PATH))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

if not batch4_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Batch 4 selected-case and difference-map validation failed.")

print("Batch 4 complete.")


,check_name,observed,expected,passed,issue
0,batch3_validation_passed,True,True,True,
1,difference_map_manifest_path_column_present,figure_path,one supported difference-map path column,True,
2,difference_map_manifest_report_overlap,355,>= 8,True,
3,difference_map_files_available,360,>= 8,True,
4,selected_cases_nonempty,18,> 0,True,
5,selected_cases_size,18,8 to 18,True,
6,selected_case_ids_unique,18,18,True,
7,selected_cases_subset_of_report,"{'selected_cases': 18, 'not_in_report': []}",{'not_in_report': []},True,
8,selected_difference_map_paths_exist,18,18,True,
9,selected_source_image_paths_exist,"{'checked_paths': 72, 'missing_paths': 0}",{'missing_paths': 0},True,


Selected cases: outputs/19_lama_report_generation/metrics/lama_report_selected_cases.csv
Batch 4 validation: outputs/19_lama_report_generation/validation/lama_report_generation_batch4_validation.csv
Stage manifest: outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json
Batch 4 complete.


In [24]:
# ============================================================
# Batch 5 - Generate Notebook Plots And HTML Report
# ============================================================

import html
import os
import re

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

try:
    from PIL import Image, ImageDraw, ImageFont, ImageOps
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

BATCH5_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch5_validation.csv"

REPORT_PLOTS_DIR = FIGURES_DIR / "plots_lama"
REPORT_CASE_PANELS_DIR = FIGURES_DIR / "selected_case_panels_lama"
REPORT_HTML_ASSETS_DIR = FIGURES_DIR / "html_assets_lama"

for directory in [REPORT_PLOTS_DIR, REPORT_CASE_PANELS_DIR, REPORT_HTML_ASSETS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REPORT_CORE_METRICS = [
    ("mse_improvement", "MSE improvement"),
    ("psnr_improvement", "PSNR improvement"),
    ("ssim_improvement", "SSIM improvement"),
    ("lpips_improvement", "LPIPS improvement"),
    ("clip_similarity_improvement", "CLIP similarity improvement"),
    ("dinov2_similarity_improvement", "DINOv2 similarity improvement"),
    ("mean_similarity_improvement", "Mean feature similarity improvement"),
]

available_report_metrics = [
    (column, label)
    for column, label in REPORT_CORE_METRICS
    if column in lama_report_df.columns
]

if not available_report_metrics:
    raise RuntimeError("Batch 5 needs at least one report metric column.")

figure_manifest_records = []

def slugify(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_") or "item"


def numeric_column(dataframe: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(dataframe[column], errors="coerce")


def format_report_number(value, decimals: int = 4) -> str:
    if value is None or pd.isna(value):
        return ""
    try:
        return f"{float(value):.{decimals}f}"
    except Exception:
        return str(value)


def path_or_none(value) -> Path | None:
    if value is None or pd.isna(value):
        return None
    value_text = str(value).strip()
    if not value_text or value_text.lower() == "nan":
        return None
    return Path(value_text)


def html_asset_path(path: Path) -> str:
    return os.path.relpath(path, start=HTML_REPORT_OUTPUT_PATH.parent).replace("\\", "/")


def dataframe_html(dataframe: pd.DataFrame, *, max_rows: int = 30, decimals: int = 4) -> str:
    display_df = dataframe.head(max_rows).copy()

    for column in display_df.columns:
        if pd.api.types.is_numeric_dtype(display_df[column]):
            display_df[column] = display_df[column].map(
                lambda value: format_report_number(value, decimals=decimals)
            )

    return display_df.to_html(index=False, escape=True, classes="report-table")


def save_plot_figure(fig, filename: str, *, title: str, section: str, description: str) -> Path:
    path = REPORT_PLOTS_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)

    figure_manifest_records.append(
        {
            "figure_type": "plot",
            "section": section,
            "title": title,
            "description": description,
            "path": str(path),
            "path_project_relative": project_relative_path(path),
            "exists": path.is_file(),
            "size_bytes": int(path.stat().st_size) if path.is_file() else 0,
        }
    )

    return path


def add_heatmap_values(ax, matrix: np.ndarray, *, decimals: int = 2):
    for y in range(matrix.shape[0]):
        for x in range(matrix.shape[1]):
            value = matrix[y, x]
            if np.isfinite(value):
                ax.text(
                    x,
                    y,
                    f"{value:.{decimals}f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black",
                )

print("Batch 5 target:")
print("Generate metric plots, selected painting panels, figure manifest, and HTML report.")
print("Available report metrics:", [label for _, label in available_report_metrics])
print("Plot output directory:", project_relative_path(REPORT_PLOTS_DIR))
print("Case panel output directory:", project_relative_path(REPORT_CASE_PANELS_DIR))

Batch 5 target:
Generate metric plots, selected painting panels, figure manifest, and HTML report.
Available report metrics: ['MSE improvement', 'PSNR improvement', 'SSIM improvement', 'LPIPS improvement', 'CLIP similarity improvement', 'DINOv2 similarity improvement', 'Mean feature similarity improvement']
Plot output directory: outputs/19_lama_report_generation/figures/plots_lama
Case panel output directory: outputs/19_lama_report_generation/figures/selected_case_panels_lama


In [25]:
# ============================================================
# Cell 28 - Metric Plots
# ============================================================

metric_summary_records = []

for column, label in available_report_metrics:
    values = numeric_column(lama_report_df, column).dropna()
    if values.empty:
        continue

    metric_summary_records.append(
        {
            "metric": column,
            "label": label,
            "cases": int(values.shape[0]),
            "mean": float(values.mean()),
            "median": float(values.median()),
            "q25": float(values.quantile(0.25)),
            "q75": float(values.quantile(0.75)),
            "min": float(values.min()),
            "max": float(values.max()),
            "positive_rate": float((values > 0).mean()),
        }
    )

metric_overview_df = pd.DataFrame(metric_summary_records)

fig, ax = plt.subplots(figsize=(11, max(4, 0.55 * len(metric_overview_df))))
plot_df = metric_overview_df.sort_values("median", ascending=True)
y_positions = np.arange(len(plot_df))

ax.barh(y_positions, plot_df["median"], color="#2563eb", alpha=0.82)
ax.errorbar(
    plot_df["median"],
    y_positions,
    xerr=[
        (plot_df["median"] - plot_df["q25"]).abs(),
        (plot_df["q75"] - plot_df["median"]).abs(),
    ],
    fmt="none",
    ecolor="#111827",
    capsize=3,
    linewidth=1,
)

ax.axvline(0, color="#6b7280", linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(plot_df["label"])
ax.set_xlabel("Improvement value; positive means restored is better than damaged")
ax.set_title("Median Metric Improvement With IQR")
ax.grid(axis="x", alpha=0.25)

metric_overview_plot_path = save_plot_figure(
    fig,
    "metric_median_iqr_overview_lama.png",
    title="Median metric improvement with IQR",
    section="metric_overview",
    description="Median and interquartile range for each available report metric.",
)

positive_rate_metric_columns = [column for column, _ in available_report_metrics]

if "mask_type" in lama_report_df.columns:
    positive_rate_by_mask_records = []

    for mask_type, group_df in lama_report_df.groupby("mask_type", dropna=False):
        record = {"mask_type": str(mask_type), "cases": int(len(group_df))}
        for column in positive_rate_metric_columns:
            values = numeric_column(group_df, column).dropna()
            record[column] = float((values > 0).mean()) if not values.empty else np.nan
        positive_rate_by_mask_records.append(record)

    positive_rate_by_mask_df = pd.DataFrame(positive_rate_by_mask_records).set_index("mask_type")
    heatmap_df = positive_rate_by_mask_df[positive_rate_metric_columns]

    fig, ax = plt.subplots(figsize=(max(9, 1.2 * len(heatmap_df.columns)), max(4.5, 0.55 * len(heatmap_df))))
    matrix = heatmap_df.to_numpy(dtype=float)

    image = ax.imshow(matrix, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels(
        [dict(available_report_metrics).get(column, column) for column in heatmap_df.columns],
        rotation=35,
        ha="right",
    )
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    ax.set_title("Positive Improvement Rate By Mask Type")
    add_heatmap_values(ax, matrix, decimals=2)
    fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03, label="Positive improvement rate")

    positive_rate_plot_path = save_plot_figure(
        fig,
        "positive_improvement_rate_by_mask_lama.png",
        title="Positive improvement rate by mask type",
        section="mask_sensitivity",
        description="Share of cases where each metric improves after LaMa restoration.",
    )

if {"dataset_name", "mask_type", "mean_similarity_improvement"}.issubset(lama_report_df.columns):
    dataset_mask_heatmap_df = (
        lama_report_df
        .pivot_table(
            index="dataset_name",
            columns="mask_type",
            values="mean_similarity_improvement",
            aggfunc="median",
        )
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(dataset_mask_heatmap_df.columns)), max(4, 0.7 * len(dataset_mask_heatmap_df))))
    matrix = dataset_mask_heatmap_df.to_numpy(dtype=float)
    finite_values = matrix[np.isfinite(matrix)]
    vmax = float(np.nanmax(np.abs(finite_values))) if finite_values.size else 1.0

    image = ax.imshow(matrix, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_xticks(np.arange(len(dataset_mask_heatmap_df.columns)))
    ax.set_xticklabels(dataset_mask_heatmap_df.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(dataset_mask_heatmap_df.index)))
    ax.set_yticklabels(dataset_mask_heatmap_df.index)
    ax.set_title("Median Mean Feature Similarity Improvement By Dataset And Mask")
    add_heatmap_values(ax, matrix, decimals=3)
    fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03, label="Median improvement")

    dataset_mask_heatmap_path = save_plot_figure(
        fig,
        "dataset_mask_mean_similarity_heatmap_lama.png",
        title="Dataset-mask mean similarity heatmap",
        section="dataset_mask",
        description="Median mean feature-similarity improvement by dataset and mask type.",
    )

if {"mask_type", "lpips_improvement"}.issubset(lama_report_df.columns):
    grouped_values = []
    labels = []

    for mask_type, group_df in lama_report_df.groupby("mask_type", dropna=False):
        values = numeric_column(group_df, "lpips_improvement").dropna()
        if not values.empty:
            labels.append(str(mask_type))
            grouped_values.append(values.to_numpy())

    if grouped_values:
        fig, ax = plt.subplots(figsize=(max(9, 0.8 * len(grouped_values)), 5))
        try:
            ax.boxplot(grouped_values, tick_labels=labels, showfliers=False)
        except TypeError:
            ax.boxplot(grouped_values, labels=labels, showfliers=False)

        ax.axhline(0, color="#6b7280", linewidth=1)
        ax.set_title("LPIPS Improvement Distribution By Mask Type")
        ax.set_ylabel("LPIPS improvement")
        ax.tick_params(axis="x", rotation=35)
        ax.grid(axis="y", alpha=0.25)

        lpips_boxplot_path = save_plot_figure(
            fig,
            "lpips_improvement_by_mask_type_lama.png",
            title="LPIPS improvement by mask type",
            section="lpips",
            description="Distribution of local LPIPS improvement across mask types.",
        )

correlation_columns = [
    column for column, _ in available_report_metrics
    if column in lama_report_df.columns
]

if len(correlation_columns) >= 2:
    correlation_plot_df = (
        lama_report_df[correlation_columns]
        .apply(pd.to_numeric, errors="coerce")
        .corr()
    )

    fig, ax = plt.subplots(figsize=(max(7, 0.8 * len(correlation_plot_df.columns)), max(6, 0.7 * len(correlation_plot_df))))
    matrix = correlation_plot_df.to_numpy(dtype=float)

    image = ax.imshow(matrix, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(np.arange(len(correlation_plot_df.columns)))
    ax.set_xticklabels(
        [dict(available_report_metrics).get(column, column) for column in correlation_plot_df.columns],
        rotation=35,
        ha="right",
    )
    ax.set_yticks(np.arange(len(correlation_plot_df.index)))
    ax.set_yticklabels([dict(available_report_metrics).get(column, column) for column in correlation_plot_df.index])
    ax.set_title("Metric Correlation Heatmap")
    add_heatmap_values(ax, matrix, decimals=2)
    fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03, label="Pearson correlation")

    correlation_heatmap_path = save_plot_figure(
        fig,
        "metric_correlation_heatmap_lama.png",
        title="Metric correlation heatmap",
        section="metric_agreement",
        description="Correlation between classical, LPIPS, and feature-similarity improvement metrics.",
    )

mask_area_column = first_existing_column(
    lama_report_df,
    [
        "mask_area_fraction",
        "mask_area_ratio",
        "mask_fraction",
        "mask_area_pct",
        "mask_area_percent",
        "mask_pixel_fraction",
        "mask_area_pixels",
        "mask_pixel_count",
    ],
)

scatter_metric_column = first_existing_column(
    lama_report_df,
    [
        "lpips_improvement",
        "mean_similarity_improvement",
        "dinov2_similarity_improvement",
    ],
)

if mask_area_column is not None and scatter_metric_column is not None:
    scatter_df = lama_report_df[[mask_area_column, scatter_metric_column, "mask_type"]].copy()
    scatter_df[mask_area_column] = pd.to_numeric(scatter_df[mask_area_column], errors="coerce")
    scatter_df[scatter_metric_column] = pd.to_numeric(scatter_df[scatter_metric_column], errors="coerce")
    scatter_df = scatter_df.dropna(subset=[mask_area_column, scatter_metric_column])

    if not scatter_df.empty:
        fig, ax = plt.subplots(figsize=(8, 5))

        for mask_type, group_df in scatter_df.groupby("mask_type", dropna=False):
            ax.scatter(
                group_df[mask_area_column],
                group_df[scatter_metric_column],
                alpha=0.65,
                s=22,
                label=str(mask_type),
            )

        ax.axhline(0, color="#6b7280", linewidth=1)
        ax.set_xlabel(mask_area_column)
        ax.set_ylabel(scatter_metric_column)
        ax.set_title("Damage Size Sensitivity")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, frameon=False, ncol=2)

        mask_area_scatter_path = save_plot_figure(
            fig,
            "mask_area_vs_metric_improvement_lama.png",
            title="Mask area versus metric improvement",
            section="damage_size_sensitivity",
            description=f"Relationship between {mask_area_column} and {scatter_metric_column}.",
        )

if {"case_id", "mean_similarity_improvement", "lpips_improvement"}.issubset(selected_cases_df.columns):
    profile_df = selected_cases_df.copy()
    profile_df["case_label"] = profile_df["case_id"].astype(str).str.replace("lama_", "", regex=False)
    profile_df = profile_df.tail(18)

    y_positions = np.arange(len(profile_df))
    fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(profile_df))))

    ax.barh(
        y_positions - 0.18,
        pd.to_numeric(profile_df["mean_similarity_improvement"], errors="coerce"),
        height=0.34,
        label="Mean feature similarity",
        color="#2563eb",
        alpha=0.82,
    )
    ax.barh(
        y_positions + 0.18,
        pd.to_numeric(profile_df["lpips_improvement"], errors="coerce"),
        height=0.34,
        label="LPIPS",
        color="#f97316",
        alpha=0.82,
    )

    ax.axvline(0, color="#6b7280", linewidth=1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(profile_df["case_label"])
    ax.set_xlabel("Improvement value")
    ax.set_title("Selected Case Metric Profile")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(frameon=False)

    selected_profile_plot_path = save_plot_figure(
        fig,
        "selected_case_metric_profile_lama.png",
        title="Selected case metric profile",
        section="selected_cases",
        description="LPIPS and feature-similarity improvement for the selected report cases.",
    )

figure_manifest_plot_df = pd.DataFrame(figure_manifest_records)

print("Metric overview:")
display(metric_overview_df)

print("Generated plot manifest:")
display(figure_manifest_plot_df)

Metric overview:


,metric,label,cases,mean,median,q25,q75,min,max,positive_rate
0,mse_improvement,MSE improvement,355,24268.743278,25316.815063,14966.532478,34382.659424,-971.973145,54879.527777,0.873239
1,psnr_improvement,PSNR improvement,355,15.449769,17.858096,13.552017,21.377231,-30.041712,35.439348,0.873239
2,lpips_improvement,LPIPS improvement,355,0.243354,0.261275,0.169325,0.362943,-0.342957,0.601806,0.873239
3,clip_similarity_improvement,CLIP similarity improvement,355,0.121695,0.120468,0.070578,0.178230,-0.380558,0.385461,0.873239
4,dinov2_similarity_improvement,DINOv2 similarity improvement,355,0.112756,0.083322,0.039520,0.164200,-0.455814,0.673047,0.867606
5,mean_similarity_improvement,Mean feature similarity improvement,355,0.117225,0.106813,0.066266,0.174731,-0.418186,0.457823,0.873239


Generated plot manifest:


,figure_type,section,title,description,path,path_project_relative,exists,size_bytes
0,plot,metric_overview,Median metric improvement with IQR,Median and interquartile range for each available report metric.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\metric_median_iqr_overview_lama.png,outputs/19_lama_report_generation/figures/plots_lama/metric_median_iqr_overview_lama.png,True,59386
1,plot,mask_sensitivity,Positive improvement rate by mask type,Share of cases where each metric improves after LaMa restoration.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\positive_improvement_rate_by_mask_lama.png,outputs/19_lama_report_generation/figures/plots_lama/positive_improvement_rate_by_mask_lama.png,True,116743
2,plot,dataset_mask,Dataset-mask mean similarity heatmap,Median mean feature-similarity improvement by dataset and mask type.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\dataset_mask_mean_similarity_heatmap_lama.png,outputs/19_lama_report_generation/figures/plots_lama/dataset_mask_mean_similarity_heatmap_lama.png,True,61582
3,plot,lpips,LPIPS improvement by mask type,Distribution of local LPIPS improvement across mask types.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\lpips_improvement_by_mask_type_lama.png,outputs/19_lama_report_generation/figures/plots_lama/lpips_improvement_by_mask_type_lama.png,True,52497
4,plot,metric_agreement,Metric correlation heatmap,"Correlation between classical, LPIPS, and feature-similarity improvement metrics.",D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\metric_correlation_heatmap_lama.png,outputs/19_lama_report_generation/figures/plots_lama/metric_correlation_heatmap_lama.png,True,157022
5,plot,damage_size_sensitivity,Mask area versus metric improvement,Relationship between mask_area_pixels and lpips_improvement.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\mask_area_vs_metric_improvement_lama.png,outputs/19_lama_report_generation/figures/plots_lama/mask_area_vs_metric_improvement_lama.png,True,111499
6,plot,selected_cases,Selected case metric profile,LPIPS and feature-similarity improvement for the selected report cases.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\selected_case_metric_profile_lama.png,outputs/19_lama_report_generation/figures/plots_lama/selected_case_metric_profile_lama.png,True,159090


In [26]:
# ============================================================
# Cell 29 - Selected Painting Panels
# ============================================================

if not PIL_AVAILABLE:
    raise RuntimeError("Pillow is required to generate selected painting panels.")

case_panel_records = []

PANEL_IMAGE_ROLES = [
    ("clean_path_resolved", "Clean"),
    ("damaged_path_resolved", "Damaged"),
    ("restored_path_resolved", "LaMa restored"),
    ("mask_path_resolved", "Mask"),
    ("error_map_figure_path", "Difference map"),
]

panel_font = ImageFont.load_default()

def draw_text(draw: ImageDraw.ImageDraw, position: tuple[int, int], text: str, fill=(17, 24, 39)):
    draw.text(position, str(text), fill=fill, font=panel_font)


def make_placeholder(tile_size: tuple[int, int], label: str) -> Image.Image:
    image = Image.new("RGB", tile_size, color=(243, 244, 246))
    draw = ImageDraw.Draw(image)
    draw.rectangle([0, 0, tile_size[0] - 1, tile_size[1] - 1], outline=(209, 213, 219), width=2)
    draw_text(draw, (12, tile_size[1] // 2 - 8), f"Missing: {label}", fill=(154, 52, 18))
    return image


def load_panel_image(path_value, tile_size: tuple[int, int], label: str) -> tuple[Image.Image, bool, str]:
    path = path_or_none(path_value)

    if path is None or not path.is_file():
        return make_placeholder(tile_size, label), False, "" if path is None else project_relative_path(path)

    image = Image.open(path).convert("RGB")
    image = ImageOps.contain(image, tile_size, method=Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", tile_size, color=(255, 255, 255))
    offset = ((tile_size[0] - image.width) // 2, (tile_size[1] - image.height) // 2)
    canvas.paste(image, offset)

    return canvas, True, project_relative_path(path)


def create_case_panel(row: pd.Series) -> dict:
    case_id = str(row.get("case_id", "case"))
    title = str(row.get("title", "Untitled"))
    mask_type = str(row.get("mask_type", ""))
    dataset_name = str(row.get("dataset_name", ""))

    tile_size = (260, 220)
    header_height = 72
    label_height = 30
    margin = 14

    panel_width = margin + len(PANEL_IMAGE_ROLES) * (tile_size[0] + margin)
    panel_height = header_height + tile_size[1] + label_height + margin

    panel = Image.new("RGB", (panel_width, panel_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(panel)

    header = f"{case_id} | {title}"
    subtitle = f"Dataset: {dataset_name} | Mask: {mask_type}"
    draw_text(draw, (margin, 12), header[:150])
    draw_text(draw, (margin, 36), subtitle[:150], fill=(75, 85, 99))

    role_exists = {}

    for index, (column, label) in enumerate(PANEL_IMAGE_ROLES):
        x = margin + index * (tile_size[0] + margin)
        y = header_height

        image, exists, source_path = load_panel_image(row.get(column, None), tile_size, label)
        panel.paste(image, (x, y))

        draw.rectangle(
            [x, y, x + tile_size[0] - 1, y + tile_size[1] - 1],
            outline=(209, 213, 219),
            width=1,
        )
        draw_text(draw, (x, y + tile_size[1] + 8), label)

        role_exists[column] = exists
        role_exists[f"{column}_project_relative"] = source_path

    output_path = REPORT_CASE_PANELS_DIR / f"{slugify(case_id)}_selected_case_panel_lama.png"
    panel.save(output_path, quality=95)

    return {
        "figure_type": "selected_case_panel",
        "section": "selected_cases",
        "case_id": case_id,
        "painting_id": row.get("painting_id", ""),
        "title": title,
        "dataset_name": dataset_name,
        "mask_type": mask_type,
        "path": str(output_path),
        "path_project_relative": project_relative_path(output_path),
        "exists": output_path.is_file(),
        "size_bytes": int(output_path.stat().st_size) if output_path.is_file() else 0,
        **role_exists,
    }


for _, selected_row in selected_cases_df.iterrows():
    case_panel_records.append(create_case_panel(selected_row))

case_panel_manifest_df = pd.DataFrame(case_panel_records)

for _, record in case_panel_manifest_df.iterrows():
    figure_manifest_records.append(
        {
            "figure_type": "selected_case_panel",
            "section": "selected_cases",
            "title": record.get("case_id", ""),
            "description": "Clean, damaged, restored, mask, and Notebook 16 difference-map panel.",
            "case_id": record.get("case_id", ""),
            "path": record.get("path", ""),
            "path_project_relative": record.get("path_project_relative", ""),
            "exists": bool(record.get("exists", False)),
            "size_bytes": int(record.get("size_bytes", 0)),
        }
    )

figure_manifest_df = pd.DataFrame(figure_manifest_records)
REPORT_FIGURE_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
figure_manifest_df.to_csv(REPORT_FIGURE_MANIFEST_PATH, index=False)

print("Selected case panel manifest:")
display(case_panel_manifest_df)

print("Batch 5 figure manifest:")
display(figure_manifest_df)

Selected case panel manifest:


,figure_type,section,case_id,painting_id,title,dataset_name,mask_type,path,path_project_relative,exists,size_bytes,clean_path_resolved,clean_path_resolved_project_relative,damaged_path_resolved,damaged_path_resolved_project_relative,restored_path_resolved,restored_path_resolved_project_relative,mask_path_resolved,mask_path_resolved_project_relative,error_map_figure_path,error_map_figure_path_project_relative
0,selected_case_panel,selected_cases,mask_robustness__p001__loss_large__variant_03,p001,Juan de Pareja,mask_robustness,nan,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\mask_robustness_p001_loss_large_variant_03_selected_case_pane...,outputs/19_lama_report_generation/figures/selected_case_panels_lama/mask_robustness_p001_loss_large_variant_03_selected_case_panel_lama.png,True,134799,True,data/processed/clean/p001_clean.png,True,data/processed/damaged/mask_robustness/p001__loss_large__variant_03_damaged.png,True,data/processed/restored/lama/mask_robustness/mask_robustness__p001__loss_large__variant_03_restored_lama.png,True,data/processed/masks/mask_robustness/p001__loss_large__variant_03_mask.png,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_mask_robustness_mask_robustness_p001_loss_large_variant_03_lama_difference_maps.png
1,selected_case_panel,selected_cases,canonical__p050_mixed_damage,p050,Landscape with a Sunlit Stream,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\canonical_p050_mixed_damage_selected_case_panel_lama.png,outputs/19_lama_report_generation/figures/selected_case_panels_lama/canonical_p050_mixed_damage_selected_case_panel_lama.png,True,192416,True,data/processed/clean/p050_clean.png,True,data/processed/masked/p050_mixed_damage_damaged.png,True,data/processed/restored/lama/canonical/canonical__p050_mixed_damage_restored_lama.png,True,data/processed/masks/p050_mixed_damage_mask.png,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p050_mixed_damage_lama_difference_maps.png
2,selected_case_panel,selected_cases,canonical__p001_mixed_damage,p001,Juan de Pareja,canonical,mixed_damage,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\canonical_p001_mixed_damage_selected_case_panel_lama.png,outputs/19_lama_report_generation/figures/selected_case_panels_lama/canonical_p001_mixed_damage_selected_case_panel_lama.png,True,175683,True,data/processed/clean/p001_clean.png,True,data/processed/masked/p001_mixed_damage_damaged.png,True,data/processed/restored/lama/canonical/canonical__p001_mixed_damage_restored_lama.png,True,data/processed/masks/p001_mixed_damage_mask.png,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png
3,selected_case_panel,selected_cases,canonical__p027_loss_large,p027,Italian Landscape with the Ponte Lucano over the Aniene River,canonical,loss_large,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\canonical_p027_loss_large_selected_case_panel_lama.png,outputs/19_lama_report_generation/figures/selected_case_panels_lama/canonical_p027_loss_large_selected_case_panel_lama.png,True,150364,True,data/processed/clean/p027_clean.png,True,data/processed/masked/p027_loss_large_damaged.png,True,data/processed/restored/lama/canonical/canonical__p027_loss_large_restored_lama.png,True,data/processed/masks/p027_loss_large_mask.png,True,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p027_loss_large_lama_difference_maps.png
4,selected_case_panel,selected_cases,canonical__p006_scratch_thin,p006,Boy with a Sword,canonical,scratch_thin,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\canonic

Batch 5 figure manifest:


,figure_type,section,title,description,path,path_project_relative,exists,size_bytes,case_id
0,plot,metric_overview,Median metric improvement with IQR,Median and interquartile range for each available report metric.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\metric_median_iqr_overview_lama.png,outputs/19_lama_report_generation/figures/plots_lama/metric_median_iqr_overview_lama.png,True,59386,NaN
1,plot,mask_sensitivity,Positive improvement rate by mask type,Share of cases where each metric improves after LaMa restoration.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\positive_improvement_rate_by_mask_lama.png,outputs/19_lama_report_generation/figures/plots_lama/positive_improvement_rate_by_mask_lama.png,True,116743,NaN
2,plot,dataset_mask,Dataset-mask mean similarity heatmap,Median mean feature-similarity improvement by dataset and mask type.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\dataset_mask_mean_similarity_heatmap_lama.png,outputs/19_lama_report_generation/figures/plots_lama/dataset_mask_mean_similarity_heatmap_lama.png,True,61582,NaN
3,plot,lpips,LPIPS improvement by mask type,Distribution of local LPIPS improvement across mask types.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\lpips_improvement_by_mask_type_lama.png,outputs/19_lama_report_generation/figures/plots_lama/lpips_improvement_by_mask_type_lama.png,True,52497,NaN
4,plot,metric_agreement,Metric correlation heatmap,"Correlation between classical, LPIPS, and feature-similarity improvement metrics.",D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\metric_correlation_heatmap_lama.png,outputs/19_lama_report_generation/figures/plots_lama/metric_correlation_heatmap_lama.png,True,157022,NaN
5,plot,damage_size_sensitivity,Mask area versus metric improvement,Relationship between mask_area_pixels and lpips_improvement.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\mask_area_vs_metric_improvement_lama.png,outputs/19_lama_report_generation/figures/plots_lama/mask_area_vs_metric_improvement_lama.png,True,111499,NaN
6,plot,selected_cases,Selected case metric profile,LPIPS and feature-similarity improvement for the selected report cases.,D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\plots_lama\selected_case_metric_profile_lama.png,outputs/19_lama_report_generation/figures/plots_lama/selected_case_metric_profile_lama.png,True,159090,NaN
7,selected_case_panel,selected_cases,mask_robustness__p001__loss_large__variant_03,"Clean, damaged, restored, mask, and Notebook 16 difference-map panel.",D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\mask_robustness_p001_loss_large_variant_03_selected_case_pane...,outputs/19_lama_report_generation/figures/selected_case_panels_lama/mask_robustness_p001_loss_large_variant_03_selected_case_panel_lama.png,True,134799,mask_robustness__p001__loss_large__variant_03
8,selected_case_panel,selected_cases,canonical__p050_mixed_damage,"Clean, damaged, restored, mask, and Notebook 16 difference-map panel.",D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\canonical_p050_mixed_damage_selected_case_panel_lama.png,outputs/19_lama_report_generation/figures/selected_case_panels_lama/canonical_p050_mixed_damage_selected_case_panel_lama.png,True,192416,canonical__p050_mixed_damage
9,selected_case_panel,selected_cases,canonical__p001_mixed_damage,"Clean, damaged, restored, mask, and Notebook 16 difference-map panel.",D:\Masters\FH\Thesis\painting-restoration-eval\outputs\19_lama_report_generation\figures\selected_case_panels_lama\canonical_p001_mixed_damage_selected_ca

In [27]:
# ============================================================
# Cell 30 - Report Tables And Short Findings
# ============================================================

report_status_cards_df = pd.DataFrame(
    [
        {"item": "Report cases", "value": int(len(lama_report_df))},
        {"item": "Selected visual cases", "value": int(len(selected_cases_df))},
        {
            "item": "Selected cases with difference maps",
            "value": int(selected_cases_df["error_map_figure_path_exists"].sum())
            if "error_map_figure_path_exists" in selected_cases_df.columns
            else 0,
        },
        {
            "item": "Datasets covered",
            "value": int(lama_report_df["dataset_name"].nunique())
            if "dataset_name" in lama_report_df.columns
            else 0,
        },
        {
            "item": "Mask types covered",
            "value": int(lama_report_df["mask_type"].nunique())
            if "mask_type" in lama_report_df.columns
            else 0,
        },
        {
            "item": "Painting categories covered",
            "value": int(lama_report_df["category"].nunique())
            if "category" in lama_report_df.columns
            else 0,
        },
        {"item": "Generated plots", "value": int((figure_manifest_df["figure_type"] == "plot").sum())},
        {
            "item": "Generated selected-case panels",
            "value": int((figure_manifest_df["figure_type"] == "selected_case_panel").sum()),
        },
    ]
)

finding_notes = []

def append_extreme_finding(
    dataframe: pd.DataFrame,
    group_column: str,
    metric_column: str,
    label: str,
):
    if group_column not in dataframe.columns or metric_column not in dataframe.columns:
        return

    grouped = (
        dataframe
        .assign(_metric=pd.to_numeric(dataframe[metric_column], errors="coerce"))
        .dropna(subset=["_metric"])
        .groupby(group_column, dropna=False)["_metric"]
        .median()
        .sort_values(ascending=False)
    )

    if grouped.empty:
        return

    strongest_group = grouped.index[0]
    weakest_group = grouped.index[-1]

    finding_notes.append(
        f"{label}: strongest median {metric_column} appears in {strongest_group} "
        f"({grouped.iloc[0]:.4f}); weakest appears in {weakest_group} ({grouped.iloc[-1]:.4f})."
    )

append_extreme_finding(lama_report_df, "mask_type", "lpips_improvement", "Mask-level perceptual behavior")
append_extreme_finding(lama_report_df, "mask_type", "mean_similarity_improvement", "Mask-level feature behavior")
append_extreme_finding(lama_report_df, "dataset_name", "mean_similarity_improvement", "Dataset-level feature behavior")
append_extreme_finding(lama_report_df, "category", "mean_similarity_improvement", "Category-level feature behavior")

if "metric_candidate" in selected_cases_df.columns:
    finding_notes.append(
        f"{int(selected_cases_df['metric_candidate'].sum())} selected cases were direct metric-priority cases; "
        f"the rest preserve dataset, mask, category, and visual coverage."
    )

report_findings_df = pd.DataFrame(
    [{"finding": note} for note in finding_notes]
)

selected_case_report_columns = [
    column
    for column in [
        "case_id",
        "painting_id",
        "title",
        "dataset_name",
        "category",
        "mask_type",
        "selection_reason",
        "mse_improvement",
        "lpips_improvement",
        "clip_similarity_improvement",
        "dinov2_similarity_improvement",
        "mean_similarity_improvement",
        "error_map_figure_path_project_relative",
    ]
    if column in selected_cases_df.columns
]

selected_case_report_table_df = selected_cases_df[selected_case_report_columns].copy()

print("Report status cards:")
display(report_status_cards_df)

print("Metric overview:")
display(metric_overview_df)

print("Short report findings:")
display(report_findings_df)

print("Selected case report table:")
display(selected_case_report_table_df)

Report status cards:


,item,value
0,Report cases,355
1,Selected visual cases,18
2,Selected cases with difference maps,18
3,Datasets covered,4
4,Mask types covered,4
5,Painting categories covered,5
6,Generated plots,7
7,Generated selected-case panels,18


Metric overview:


,metric,label,cases,mean,median,q25,q75,min,max,positive_rate
0,mse_improvement,MSE improvement,355,24268.743278,25316.815063,14966.532478,34382.659424,-971.973145,54879.527777,0.873239
1,psnr_improvement,PSNR improvement,355,15.449769,17.858096,13.552017,21.377231,-30.041712,35.439348,0.873239
2,lpips_improvement,LPIPS improvement,355,0.243354,0.261275,0.169325,0.362943,-0.342957,0.601806,0.873239
3,clip_similarity_improvement,CLIP similarity improvement,355,0.121695,0.120468,0.070578,0.178230,-0.380558,0.385461,0.873239
4,dinov2_similarity_improvement,DINOv2 similarity improvement,355,0.112756,0.083322,0.039520,0.164200,-0.455814,0.673047,0.867606
5,mean_similarity_improvement,Mean feature similarity improvement,355,0.117225,0.106813,0.066266,0.174731,-0.418186,0.457823,0.873239


Short report findings:


,finding
0,Mask-level perceptual behavior: strongest median lpips_improvement appears in loss_large (0.3298); weakest appears in loss_small (0.1834).
1,Mask-level feature behavior: strongest median mean_similarity_improvement appears in loss_large (0.1803); weakest appears in loss_small (0.0775).
2,Dataset-level feature behavior: strongest median mean_similarity_improvement appears in damage_size (0.1523); weakest appears in synthetic_degradation (-0.0044).
3,Category-level feature behavior: strongest median mean_similarity_improvement appears in landscape_natural (0.1568); weakest appears in abstraction_surrealism (0.0782).
4,"18 selected cases were direct metric-priority cases; the rest preserve dataset, mask, category, and visual coverage."


Selected case report table:


,case_id,painting_id,title,dataset_name,category,mask_type,selection_reason,mse_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement,mean_similarity_improvement,error_map_figure_path_project_relative
0,mask_robustness__p001__loss_large__variant_03,p001,Juan de Pareja,mask_robustness,portrait_figure,NaN,Representative high-scoring category example; Representative high-scoring dataset example | Notebook 16 difference-map figure available,48786.026550,0.516388,0.259772,0.443252,0.351512,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_mask_robustness_mask_robustness_p001_loss_large_variant_03_lama_difference_maps.png
1,canonical__p050_mixed_damage,p050,Landscape with a Sunlit Stream,canonical,high_texture_brushwork,mixed_damage,Representative high-scoring category example; Representative high-scoring dataset example; Strongest local feature-similarity improvement | Notebook 16 difference-map figure av...,40351.888306,0.422922,0.242599,0.673047,0.457823,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p050_mixed_damage_lama_difference_maps.png
2,canonical__p001_mixed_damage,p001,Juan de Pareja,canonical,portrait_figure,mixed_damage,Slowest restoration runtime | Notebook 16 difference-map figure available,50582.575195,0.479921,0.286900,0.215196,0.251048,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p001_mixed_damage_lama_difference_maps.png
3,canonical__p027_loss_large,p027,Italian Landscape with the Ponte Lucano over the Aniene River,canonical,architecture_structured,loss_large,Representative high-scoring category example | Notebook 16 difference-map figure available,29947.750305,0.515375,0.321936,0.329906,0.325921,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p027_loss_large_lama_difference_maps.png
4,canonical__p006_scratch_thin,p006,Boy with a Sword,canonical,portrait_figure,scratch_thin,Strongest local MSE improvement | Notebook 16 difference-map figure available,53519.212227,0.415661,0.239234,0.210861,0.225047,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p006_scratch_thin_lama_difference_maps.png
5,canonical__p013_mixed_damage,p013,Landscape with a Village in the Distance,canonical,landscape_natural,mixed_damage,Representative high-scoring category example; Strongest local feature-similarity improvement | Notebook 16 difference-map figure available,38569.472046,0.370276,0.167663,0.577497,0.372580,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p013_mixed_damage_lama_difference_maps.png
6,canonical__p044_mixed_damage,p044,Versailles,canonical,high_texture_brushwork,mixed_damage,Strongest local feature-similarity improvement | Notebook 16 difference-map figure available,28698.513367,0.353440,0.202063,0.580601,0.391332,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p044_mixed_damage_lama_difference_maps.png
7,canonical__p011_loss_large,p011,View of Haarlem and the Haarlemmer Meer,canonical,landscape_natural,loss_large,Strongest local LPIPS improvement | Notebook 16 difference-map figure available,18223.183861,0.560803,0.253034,0.308888,0.280961,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p011_loss_large_lama_difference_maps.png
8,canonical__p014_mixed_damage,p014,Landscape on a River,canonical,landscape_natural,mixed_damage,Strongest local feature-similarity improvement | Notebook 16 difference-map figure available,29267.156128,0.312184,0.197791,0.513790,0.355790,outputs/16_lama_difference_maps/figures/all_cases/all_cases/lama_canonical_canonical_p014_mixed_damage_lama_difference_maps.png
9,canonical__p001_scratch_thin,p001,Juan de Pareja,canonical,portrait_figure,scratch_thin,Slowest restoration runtime | Notebook 16 difference-map figure available,49721.899544,0.522427,0.202154,0.100681,0.151418,outputs/16_lama_

In [28]:
# ============================================================
# Cell 31 - Build Standalone HTML Report
# ============================================================

plot_manifest_df = figure_manifest_df.loc[
    figure_manifest_df["figure_type"].astype(str).eq("plot")
].copy()

panel_manifest_df = case_panel_manifest_df.copy()

selected_cases_for_html_df = selected_cases_df.merge(
    panel_manifest_df[["case_id", "path", "path_project_relative"]].rename(
        columns={
            "path": "case_panel_path",
            "path_project_relative": "case_panel_path_project_relative",
        }
    ),
    on="case_id",
    how="left",
)

def image_figure_html(path_value, caption: str, alt: str) -> str:
    path = path_or_none(path_value)
    if path is None or not path.is_file():
        return f"<div class='missing'>Missing image: {html.escape(str(path_value))}</div>"

    return f"""
    <figure>
        <img src="{html.escape(html_asset_path(path))}" alt="{html.escape(alt)}">
        <figcaption>{html.escape(caption)}</figcaption>
    </figure>
    """


plot_gallery_html = "\n".join(
    f"""
    <article class="figure-card">
        {image_figure_html(row.get("path", ""), row.get("description", ""), row.get("title", ""))}
        <h3>{html.escape(str(row.get("title", "")))}</h3>
    </article>
    """
    for _, row in plot_manifest_df.iterrows()
)

def selected_case_html(row: pd.Series) -> str:
    metric_rows = []

    for column, label in available_report_metrics:
        if column in row.index:
            metric_rows.append(
                {
                    "metric": label,
                    "value": format_report_number(row.get(column), decimals=5),
                }
            )

    metric_table_html = dataframe_html(pd.DataFrame(metric_rows), max_rows=20, decimals=5)

    panel_html = image_figure_html(
        row.get("case_panel_path", ""),
        "Clean, damaged, restored, mask, and difference-map evidence.",
        str(row.get("case_id", "")),
    )

    return f"""
    <section class="case-card">
        <h3>{html.escape(str(row.get("case_id", "")))}: {html.escape(str(row.get("title", "")))}</h3>
        <p class="case-meta">
            Dataset: <b>{html.escape(str(row.get("dataset_name", "")))}</b> |
            Category: <b>{html.escape(str(row.get("category", "")))}</b> |
            Mask: <b>{html.escape(str(row.get("mask_type", "")))}</b>
        </p>
        <p><b>Why this case is included:</b> {html.escape(str(row.get("selection_reason", "")))}</p>
        {panel_html}
        {metric_table_html}
    </section>
    """

selected_cases_html = "\n".join(
    selected_case_html(row)
    for _, row in selected_cases_for_html_df.iterrows()
)

finding_html = (
    "<ul>" + "\n".join(f"<li>{html.escape(str(note))}</li>" for note in finding_notes) + "</ul>"
    if finding_notes
    else "<p>No compact finding notes were generated.</p>"
)

html_report = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>{html.escape(REPORT_TITLE)}</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 34px;
            background: #f8fafc;
            color: #111827;
            line-height: 1.5;
        }}
        h1 {{
            margin-bottom: 6px;
            border-bottom: 2px solid #111827;
            padding-bottom: 10px;
        }}
        h2 {{
            margin-top: 32px;
        }}
        h3 {{
            margin-bottom: 8px;
        }}
        .subtitle {{
            color: #4b5563;
            margin-top: 0;
        }}
        .section {{
            background: #ffffff;
            border: 1px solid #d1d5db;
            border-radius: 8px;
            padding: 20px;
            margin: 20px 0;
        }}
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(320px, 1fr));
            gap: 18px;
        }}
        .figure-card,
        .case-card {{
            border: 1px solid #d1d5db;
            border-radius: 8px;
            padding: 14px;
            background: #ffffff;
            page-break-inside: avoid;
        }}
        figure {{
            margin: 0 0 12px 0;
        }}
        img {{
            max-width: 100%;
            height: auto;
            border: 1px solid #e5e7eb;
            background: #f3f4f6;
        }}
        figcaption {{
            font-size: 12px;
            color: #4b5563;
            margin-top: 6px;
        }}
        .report-table {{
            border-collapse: collapse;
            width: 100%;
            margin: 12px 0 18px 0;
            font-size: 13px;
        }}
        .report-table th,
        .report-table td {{
            border: 1px solid #d1d5db;
            padding: 7px;
            text-align: left;
            vertical-align: top;
        }}
        .report-table th {{
            background: #e5e7eb;
        }}
        .case-meta {{
            color: #374151;
        }}
        .note {{
            background: #eff6ff;
            border: 1px solid #bfdbfe;
            border-radius: 6px;
            padding: 12px;
        }}
        .missing {{
            color: #9a3412;
            font-style: italic;
        }}
    </style>
</head>
<body>
    <h1>{html.escape(REPORT_TITLE)}</h1>
    <p class="subtitle">
        Notebook 19 synthesis report for LaMa restoration: paintings, plots, metrics, and compact interpretation.
    </p>

    <section class="section">
        <h2>Scope</h2>
        <p>
            This report consolidates the LaMa restoration dataframe, region-aware classical metrics,
            LPIPS, feature similarity, selected source-image panels, and Notebook 16 difference maps.
            Zero-control rows remain validation context and are excluded from the main diagnostic report.
        </p>
        {dataframe_html(report_status_cards_df, max_rows=20, decimals=4)}
    </section>

    <section class="section">
        <h2>Metric Overview</h2>
        <p>
            Positive values mean the restored image improved over the damaged image for the local report region.
            The plots emphasize distribution, group sensitivity, and agreement between metric families.
        </p>
        {dataframe_html(metric_overview_df, max_rows=30, decimals=5)}
    </section>

    <section class="section">
        <h2>Compact Findings</h2>
        {finding_html}
    </section>

    <section class="section">
        <h2>Plot Gallery</h2>
        <div class="grid">
            {plot_gallery_html}
        </div>
    </section>

    <section class="section">
        <h2>Summary Tables</h2>
        <h3>Overview</h3>
        {dataframe_html(summary_overview_df, max_rows=30, decimals=4)}
        <h3>By Dataset</h3>
        {dataframe_html(summary_by_dataset_df, max_rows=30, decimals=5)}
        <h3>By Mask Type</h3>
        {dataframe_html(summary_by_mask_type_df, max_rows=30, decimals=5)}
        <h3>By Category</h3>
        {dataframe_html(summary_by_category_df, max_rows=30, decimals=5)}
        <h3>Metric Correlations</h3>
        {metric_correlation_df.round(4).to_html(classes="report-table")}
    </section>

    <section class="section">
        <h2>Selected Painting Cases</h2>
        <p class="note">
            These cases combine metric extremes, group coverage, and visual availability. Each panel shows
            the clean source, damaged input, LaMa output, mask, and Notebook 16 difference-map evidence.
        </p>
        {selected_cases_html}
    </section>

    <section class="section">
        <h2>Report Reading Notes</h2>
        <p>
            Treat this as a synthesis report, not a final thesis conclusion. The useful reading pattern is:
            inspect metric direction first, then check whether the selected painting panel supports that signal.
            Cases where metric families disagree should be carried forward into grouping and risk-flag notebooks.
        </p>
    </section>
</body>
</html>
"""

HTML_REPORT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
HTML_REPORT_OUTPUT_PATH.write_text(html_report, encoding="utf-8")

print("HTML report:", project_relative_path(HTML_REPORT_OUTPUT_PATH))
print("HTML report size bytes:", HTML_REPORT_OUTPUT_PATH.stat().st_size if HTML_REPORT_OUTPUT_PATH.is_file() else 0)

HTML report: outputs/19_lama_report_generation/reports/lama_restoration_evaluation_report.html
HTML report size bytes: 49476


In [29]:
# ============================================================
# Cell 32 - Batch 5 Validation And Stage Manifest Update
# ============================================================

figure_manifest_exists = REPORT_FIGURE_MANIFEST_PATH.is_file()
html_report_exists = HTML_REPORT_OUTPUT_PATH.is_file()

plot_count = int((figure_manifest_df["figure_type"] == "plot").sum())
case_panel_count = int((figure_manifest_df["figure_type"] == "selected_case_panel").sum())

all_figure_manifest_paths_exist = (
    bool(figure_manifest_df["exists"].astype(bool).all())
    if not figure_manifest_df.empty
    else False
)

all_case_panels_exist = (
    bool(case_panel_manifest_df["exists"].astype(bool).all())
    if not case_panel_manifest_df.empty
    else False
)

batch5_output_paths = {
    "figure_manifest": REPORT_FIGURE_MANIFEST_PATH,
    "html_report": HTML_REPORT_OUTPUT_PATH,
}

batch5_outputs_exist = {
    name: path.is_file()
    for name, path in batch5_output_paths.items()
}

batch5_outputs_nonempty = {
    name: int(path.stat().st_size) if path.is_file() else 0
    for name, path in batch5_output_paths.items()
}

batch5_validation_df = pd.DataFrame(
    [
        build_check(
            "batch4_validation_passed",
            validation_passed_from_csv(BATCH4_VALIDATION_OUTPUT_PATH),
            True,
            validation_passed_from_csv(BATCH4_VALIDATION_OUTPUT_PATH),
            "Batch 4 validation did not pass",
        ),
        build_check(
            "plots_generated",
            plot_count,
            ">= 5",
            plot_count >= 5,
            "too few report plots were generated",
        ),
        build_check(
            "selected_case_panels_generated",
            case_panel_count,
            int(len(selected_cases_df)),
            case_panel_count == int(len(selected_cases_df)),
            "not every selected case received a painting panel",
        ),
        build_check(
            "figure_manifest_exists",
            figure_manifest_exists,
            True,
            figure_manifest_exists,
            "Batch 5 figure manifest was not written",
        ),
        build_check(
            "figure_manifest_nonempty",
            int(REPORT_FIGURE_MANIFEST_PATH.stat().st_size) if figure_manifest_exists else 0,
            "> 0",
            figure_manifest_exists and REPORT_FIGURE_MANIFEST_PATH.stat().st_size > 0,
            "Batch 5 figure manifest is empty",
        ),
        build_check(
            "figure_manifest_paths_exist",
            all_figure_manifest_paths_exist,
            True,
            all_figure_manifest_paths_exist,
            "one or more Batch 5 figure paths is missing",
        ),
        build_check(
            "selected_case_panel_paths_exist",
            all_case_panels_exist,
            True,
            all_case_panels_exist,
            "one or more selected case panel paths is missing",
        ),
        build_check(
            "html_report_exists",
            html_report_exists,
            True,
            html_report_exists,
            "HTML report was not written",
        ),
        build_check(
            "html_report_nonempty",
            int(HTML_REPORT_OUTPUT_PATH.stat().st_size) if html_report_exists else 0,
            "> 0",
            html_report_exists and HTML_REPORT_OUTPUT_PATH.stat().st_size > 0,
            "HTML report is empty",
        ),
        build_check(
            "batch5_outputs_exist",
            batch5_outputs_exist,
            "all true",
            all(batch5_outputs_exist.values()),
            "one or more Batch 5 outputs is missing",
        ),
        build_check(
            "batch5_outputs_nonempty",
            batch5_outputs_nonempty,
            "all > 0",
            all(value > 0 for value in batch5_outputs_nonempty.values()),
            "one or more Batch 5 outputs is empty",
        ),
    ]
)

batch5_validation_export_df = batch5_validation_df.copy()

for column in ["observed", "expected"]:
    batch5_validation_export_df[column] = batch5_validation_export_df[column].map(json_for_csv)

BATCH5_VALIDATION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
batch5_validation_export_df.to_csv(BATCH5_VALIDATION_OUTPUT_PATH, index=False)

if "stage_manifest" not in globals() or not isinstance(stage_manifest, dict):
    stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH)

stage_manifest.setdefault("batches", {})
stage_manifest["batches"]["batch_5"] = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "generate_notebook_plots_selected_painting_panels_and_html_report",
    "validation": project_relative_path(BATCH5_VALIDATION_OUTPUT_PATH),
    "validation_passed": bool(batch5_validation_df["passed"].astype(bool).all()),
    "outputs": {
        "html_report": project_relative_path(HTML_REPORT_OUTPUT_PATH),
        "figure_manifest": project_relative_path(REPORT_FIGURE_MANIFEST_PATH),
        "plots_dir": project_relative_path(REPORT_PLOTS_DIR),
        "selected_case_panels_dir": project_relative_path(REPORT_CASE_PANELS_DIR),
    },
    "figure_counts": {
        "plots": plot_count,
        "selected_case_panels": case_panel_count,
        "total_manifest_rows": int(len(figure_manifest_df)),
    },
    "html_report_size_bytes": int(HTML_REPORT_OUTPUT_PATH.stat().st_size)
    if HTML_REPORT_OUTPUT_PATH.is_file()
    else 0,
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

display(batch5_validation_df)

print("Batch 5 validation:", project_relative_path(BATCH5_VALIDATION_OUTPUT_PATH))
print("Figure manifest:", project_relative_path(REPORT_FIGURE_MANIFEST_PATH))
print("HTML report:", project_relative_path(HTML_REPORT_OUTPUT_PATH))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

if not batch5_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Batch 5 plot and HTML report validation failed.")

print("Batch 5 complete.")

,check_name,observed,expected,passed,issue
0,batch4_validation_passed,True,True,True,
1,plots_generated,7,>= 5,True,
2,selected_case_panels_generated,18,18,True,
3,figure_manifest_exists,True,True,True,
4,figure_manifest_nonempty,11685,> 0,True,
5,figure_manifest_paths_exist,True,True,True,
6,selected_case_panel_paths_exist,True,True,True,
7,html_report_exists,True,True,True,
8,html_report_nonempty,49476,> 0,True,
9,batch5_outputs_exist,"{'figure_manifest': True, 'html_report': True}",all true,True,


Batch 5 validation: outputs/19_lama_report_generation/validation/lama_report_generation_batch5_validation.csv
Figure manifest: outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_figure_manifest.csv
HTML report: outputs/19_lama_report_generation/reports/lama_restoration_evaluation_report.html
Stage manifest: outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json
Batch 5 complete.


In [30]:
# ============================================================
# Batch 6 - Save CSVs, Final Validation, Stage Manifest, Handoff
# ============================================================

BATCH5_VALIDATION_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch5_validation.csv"

FINAL_ARTIFACT_INDEX_OUTPUT_PATH = MANIFESTS_DIR / "19_lama_report_generation_artifact_index.csv"
FINAL_TABLE_SHAPES_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_table_shapes.csv"
FINAL_BATCH_STATUS_OUTPUT_PATH = VALIDATION_DIR / "lama_report_generation_batch_status.csv"

FINAL_REPORT_STATUS_CARDS_OUTPUT_PATH = METRICS_DIR / "lama_report_status_cards.csv"
FINAL_METRIC_OVERVIEW_OUTPUT_PATH = METRICS_DIR / "lama_report_metric_overview.csv"
FINAL_FINDINGS_OUTPUT_PATH = METRICS_DIR / "lama_report_findings.csv"
FINAL_SELECTED_CASE_TABLE_OUTPUT_PATH = METRICS_DIR / "lama_report_selected_case_table.csv"
CASE_PANEL_MANIFEST_OUTPUT_PATH = REPORT_DIAGNOSTIC_FIGURES_DIR / "lama_report_selected_case_panel_manifest.csv"

for directory in [METRICS_DIR, VALIDATION_DIR, MANIFESTS_DIR, REPORT_DIAGNOSTIC_FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def dataframe_from_memory_or_csv(variable_name: str, path: Path, *, required: bool = True) -> pd.DataFrame:
    if variable_name in globals() and isinstance(globals()[variable_name], pd.DataFrame):
        dataframe = globals()[variable_name].copy()
        if required and dataframe.empty:
            raise ValueError(f"{variable_name} is empty.")
        return dataframe

    if path.is_file():
        return pd.read_csv(path, low_memory=False)

    if required:
        raise FileNotFoundError(f"Missing required dataframe and CSV: {project_relative_path(path)}")

    return pd.DataFrame()


lama_report_df = dataframe_from_memory_or_csv("lama_report_df", REPORT_DATAFRAME_OUTPUT_PATH)
selected_cases_df = dataframe_from_memory_or_csv("selected_cases_df", SELECTED_CASES_OUTPUT_PATH)
compact_summaries_df = dataframe_from_memory_or_csv("compact_summaries_df", REPORT_SUMMARIES_OUTPUT_PATH)
figure_manifest_df = dataframe_from_memory_or_csv("figure_manifest_df", REPORT_FIGURE_MANIFEST_PATH)

if "metric_correlation_df" in globals() and isinstance(metric_correlation_df, pd.DataFrame):
    metric_correlation_df = metric_correlation_df.copy()
elif REPORT_CORRELATION_OUTPUT_PATH.is_file():
    metric_correlation_df = pd.read_csv(REPORT_CORRELATION_OUTPUT_PATH, index_col=0)
else:
    raise FileNotFoundError(f"Missing metric correlation CSV: {project_relative_path(REPORT_CORRELATION_OUTPUT_PATH)}")

if "metric_overview_df" not in globals() or not isinstance(metric_overview_df, pd.DataFrame):
    fallback_metric_columns = [
        "mse_improvement",
        "psnr_improvement",
        "ssim_improvement",
        "lpips_improvement",
        "clip_similarity_improvement",
        "dinov2_similarity_improvement",
        "mean_similarity_improvement",
    ]

    metric_overview_records = []
    for column in fallback_metric_columns:
        if column not in lama_report_df.columns:
            continue

        values = pd.to_numeric(lama_report_df[column], errors="coerce").dropna()
        if values.empty:
            continue

        metric_overview_records.append(
            {
                "metric": column,
                "cases": int(values.shape[0]),
                "mean": float(values.mean()),
                "median": float(values.median()),
                "q25": float(values.quantile(0.25)),
                "q75": float(values.quantile(0.75)),
                "min": float(values.min()),
                "max": float(values.max()),
                "positive_rate": float((values > 0).mean()),
            }
        )

    metric_overview_df = pd.DataFrame(metric_overview_records)

if "report_status_cards_df" not in globals() or not isinstance(report_status_cards_df, pd.DataFrame):
    report_status_cards_df = pd.DataFrame(
        [
            {"item": "Report cases", "value": int(len(lama_report_df))},
            {"item": "Selected visual cases", "value": int(len(selected_cases_df))},
            {"item": "Generated figures", "value": int(len(figure_manifest_df))},
            {"item": "Datasets covered", "value": int(lama_report_df["dataset_name"].nunique()) if "dataset_name" in lama_report_df.columns else 0},
            {"item": "Mask types covered", "value": int(lama_report_df["mask_type"].nunique()) if "mask_type" in lama_report_df.columns else 0},
        ]
    )

if "report_findings_df" not in globals() or not isinstance(report_findings_df, pd.DataFrame):
    if "finding_notes" in globals() and isinstance(finding_notes, list):
        report_findings_df = pd.DataFrame([{"finding": note} for note in finding_notes])
    else:
        report_findings_df = pd.DataFrame(columns=["finding"])

if "selected_case_report_table_df" not in globals() or not isinstance(selected_case_report_table_df, pd.DataFrame):
    selected_case_report_columns = [
        column
        for column in [
            "case_id",
            "painting_id",
            "title",
            "dataset_name",
            "category",
            "mask_type",
            "selection_reason",
            "mse_improvement",
            "lpips_improvement",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
            "mean_similarity_improvement",
            "error_map_figure_path_project_relative",
        ]
        if column in selected_cases_df.columns
    ]
    selected_case_report_table_df = selected_cases_df[selected_case_report_columns].copy()

if "case_panel_manifest_df" not in globals() or not isinstance(case_panel_manifest_df, pd.DataFrame):
    if "figure_type" in figure_manifest_df.columns:
        case_panel_manifest_df = figure_manifest_df[
            figure_manifest_df["figure_type"].astype(str).eq("selected_case_panel")
        ].copy()
    else:
        case_panel_manifest_df = pd.DataFrame()

final_csv_outputs = {
    "report_dataframe": (lama_report_df, REPORT_DATAFRAME_OUTPUT_PATH, False),
    "selected_cases": (selected_cases_df, SELECTED_CASES_OUTPUT_PATH, False),
    "report_summaries": (compact_summaries_df, REPORT_SUMMARIES_OUTPUT_PATH, False),
    "metric_correlations": (metric_correlation_df, REPORT_CORRELATION_OUTPUT_PATH, True),
    "figure_manifest": (figure_manifest_df, REPORT_FIGURE_MANIFEST_PATH, False),
    "metric_overview": (metric_overview_df, FINAL_METRIC_OVERVIEW_OUTPUT_PATH, False),
    "report_status_cards": (report_status_cards_df, FINAL_REPORT_STATUS_CARDS_OUTPUT_PATH, False),
    "report_findings": (report_findings_df, FINAL_FINDINGS_OUTPUT_PATH, False),
    "selected_case_report_table": (selected_case_report_table_df, FINAL_SELECTED_CASE_TABLE_OUTPUT_PATH, False),
    "case_panel_manifest": (case_panel_manifest_df, CASE_PANEL_MANIFEST_OUTPUT_PATH, False),
}

saved_csv_records = []

for artifact_name, (dataframe, path, include_index) in final_csv_outputs.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(path, index=include_index)

    saved_csv_records.append(
        {
            "artifact": artifact_name,
            "path": project_relative_path(path),
            "rows": int(len(dataframe)),
            "columns": int(len(dataframe.columns)),
            "exists": path.is_file(),
            "size_bytes": int(path.stat().st_size) if path.is_file() else 0,
            "sha256": sha256_file(path) if path.is_file() else "",
        }
    )

saved_csv_manifest_df = pd.DataFrame(saved_csv_records)

table_shapes_df = pd.DataFrame(
    [
        {
            "table": artifact_name,
            "rows": int(len(dataframe)),
            "columns": int(len(dataframe.columns)),
            "path": project_relative_path(path),
        }
        for artifact_name, (dataframe, path, _) in final_csv_outputs.items()
    ]
)

table_shapes_df.to_csv(FINAL_TABLE_SHAPES_OUTPUT_PATH, index=False)

print("Final CSV outputs:")
display(saved_csv_manifest_df)

print("Final table shapes:")
display(table_shapes_df)

Final CSV outputs:


,artifact,path,rows,columns,exists,size_bytes,sha256
0,report_dataframe,outputs/19_lama_report_generation/metrics/lama_report_dataframe.csv,355,264,True,1468035,c032781d43c9cd27aeac949dad95fe80acf8680e6664b6c2df958ec345416669
1,selected_cases,outputs/19_lama_report_generation/metrics/lama_report_selected_cases.csv,18,273,True,81682,314347ee1b207bae8f0fbf4da83b13291ae24cc08d8b1d0e0ea32bcfa6eaa2cf
2,report_summaries,outputs/19_lama_report_generation/metrics/lama_report_summaries.csv,33,23,True,4428,1b690573ed7f624d076a0525c9abbea0a9425b283a4eec525a8128532b09dd99
3,metric_correlations,outputs/19_lama_report_generation/metrics/lama_report_metric_correlations.csv,8,8,True,681,30e3e1a1eb7bd259afc220cf758c0678efcd5e3055754f3acc6face8217f6d6d
4,figure_manifest,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_figure_manifest.csv,25,9,True,11685,4b997b88880998ac41424c0694984154880e219e0bbf303617e1a780a46b1e89
5,metric_overview,outputs/19_lama_report_generation/metrics/lama_report_metric_overview.csv,6,10,True,1173,13a8b792c9b3f0201af89941ad8f246248942739b565f748744552729d274b92
6,report_status_cards,outputs/19_lama_report_generation/metrics/lama_report_status_cards.csv,8,2,True,223,21de48205203f588170678af846e5a404e5aaa5c8845755d34d5e3fea0529bfe
7,report_findings,outputs/19_lama_report_generation/metrics/lama_report_findings.csv,5,1,True,749,54a76b4f13b1ca131686b9aeaa8bbd6f4dc5c46109f5d6f81a591272ce012a3e
8,selected_case_report_table,outputs/19_lama_report_generation/metrics/lama_report_selected_case_table.csv,18,13,True,8128,8c2dc8d0b166c017796b02635d4bf58094afc710ba2f10301d51858edc8f0b14
9,case_panel_manifest,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_selected_case_panel_manifest.csv,18,21,True,15948,56da89838a4fcd60839bd50c6e0cff59b117c861c4067c9f1d36df397dc0e114


Final table shapes:


,table,rows,columns,path
0,report_dataframe,355,264,outputs/19_lama_report_generation/metrics/lama_report_dataframe.csv
1,selected_cases,18,273,outputs/19_lama_report_generation/metrics/lama_report_selected_cases.csv
2,report_summaries,33,23,outputs/19_lama_report_generation/metrics/lama_report_summaries.csv
3,metric_correlations,8,8,outputs/19_lama_report_generation/metrics/lama_report_metric_correlations.csv
4,figure_manifest,25,9,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_figure_manifest.csv
5,metric_overview,6,10,outputs/19_lama_report_generation/metrics/lama_report_metric_overview.csv
6,report_status_cards,8,2,outputs/19_lama_report_generation/metrics/lama_report_status_cards.csv
7,report_findings,5,1,outputs/19_lama_report_generation/metrics/lama_report_findings.csv
8,selected_case_report_table,18,13,outputs/19_lama_report_generation/metrics/lama_report_selected_case_table.csv
9,case_panel_manifest,18,21,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_selected_case_panel_manifest.csv


In [31]:
# ============================================================
# Cell 34 - Final Validation And Manifests
# ============================================================

def validation_passed_strict(path: Path) -> bool:
    if not path.is_file():
        return False

    validation_df = pd.read_csv(path)

    if "passed" not in validation_df.columns:
        return False

    passed_values = validation_df["passed"]

    if pd.api.types.is_bool_dtype(passed_values):
        return bool(passed_values.all())

    return bool(
        passed_values
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
        .all()
    )


def path_size(path: Path) -> int:
    return int(path.stat().st_size) if path.is_file() else 0


def build_artifact_index(artifact_paths: dict[str, tuple[Path, str, bool]]) -> pd.DataFrame:
    records = []

    for artifact_name, (path, artifact_type, required) in artifact_paths.items():
        records.append(
            artifact_record(
                artifact_name,
                path,
                artifact_type=artifact_type,
                project_root=PROJECT_ROOT,
                required=required,
            )
        )

    return pd.DataFrame(records)


batch_validation_paths = {
    "batch_1": BATCH1_VALIDATION_OUTPUT_PATH,
    "batch_2": BATCH2_VALIDATION_OUTPUT_PATH,
    "batch_3": BATCH3_VALIDATION_OUTPUT_PATH,
    "batch_4": BATCH4_VALIDATION_OUTPUT_PATH,
    "batch_5": BATCH5_VALIDATION_OUTPUT_PATH,
}

batch_status_df = pd.DataFrame(
    [
        {
            "batch": batch_name,
            "validation_path": project_relative_path(path),
            "validation_exists": path.is_file(),
            "validation_size_bytes": path_size(path),
            "validation_passed": validation_passed_strict(path),
        }
        for batch_name, path in batch_validation_paths.items()
    ]
)

batch_status_df.to_csv(FINAL_BATCH_STATUS_OUTPUT_PATH, index=False)

FINAL_ARTIFACT_PATHS = {
    "report_dataframe": (REPORT_DATAFRAME_OUTPUT_PATH, "csv", True),
    "selected_cases": (SELECTED_CASES_OUTPUT_PATH, "csv", True),
    "report_summaries": (REPORT_SUMMARIES_OUTPUT_PATH, "csv", True),
    "metric_correlations": (REPORT_CORRELATION_OUTPUT_PATH, "csv", True),
    "metric_overview": (FINAL_METRIC_OVERVIEW_OUTPUT_PATH, "csv", True),
    "report_status_cards": (FINAL_REPORT_STATUS_CARDS_OUTPUT_PATH, "csv", True),
    "report_findings": (FINAL_FINDINGS_OUTPUT_PATH, "csv", False),
    "selected_case_report_table": (FINAL_SELECTED_CASE_TABLE_OUTPUT_PATH, "csv", True),
    "figure_manifest": (REPORT_FIGURE_MANIFEST_PATH, "csv", True),
    "case_panel_manifest": (CASE_PANEL_MANIFEST_OUTPUT_PATH, "csv", False),
    "table_shapes": (FINAL_TABLE_SHAPES_OUTPUT_PATH, "csv", True),
    "batch_status": (FINAL_BATCH_STATUS_OUTPUT_PATH, "csv", True),
    "html_report": (HTML_REPORT_OUTPUT_PATH, "html", True),
    "stage_manifest": (STAGE_MANIFEST_JSON_PATH, "json", True),
    "handoff_manifest": (HANDOFF_MANIFEST_JSON_PATH, "json", True),
    "final_validation": (FINAL_VALIDATION_OUTPUT_PATH, "csv", True),
}

preliminary_artifact_index_df = build_artifact_index(FINAL_ARTIFACT_PATHS)
preliminary_artifact_index_df.to_csv(FINAL_ARTIFACT_INDEX_OUTPUT_PATH, index=False)

batch_status_payload = {
    row["batch"]: bool(row["validation_passed"])
    for _, row in batch_status_df.iterrows()
}

core_output_payload = {
    artifact_name: project_relative_path(path)
    for artifact_name, (path, _, required) in FINAL_ARTIFACT_PATHS.items()
    if required and artifact_name not in {"stage_manifest", "handoff_manifest"}
}

stage_manifest = read_json(STAGE_MANIFEST_JSON_PATH) if STAGE_MANIFEST_JSON_PATH.is_file() else {}
stage_manifest.setdefault("batches", {})
stage_manifest["batches"]["batch_6"] = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "save_final_csvs_validate_outputs_and_write_handoff_manifest",
    "validation": project_relative_path(FINAL_VALIDATION_OUTPUT_PATH),
    "validation_passed": False,
    "outputs": {
        "final_validation": project_relative_path(FINAL_VALIDATION_OUTPUT_PATH),
        "handoff_manifest": project_relative_path(HANDOFF_MANIFEST_JSON_PATH),
        "artifact_index": project_relative_path(FINAL_ARTIFACT_INDEX_OUTPUT_PATH),
        "table_shapes": project_relative_path(FINAL_TABLE_SHAPES_OUTPUT_PATH),
        "batch_status": project_relative_path(FINAL_BATCH_STATUS_OUTPUT_PATH),
    },
    "batch_validation_status": batch_status_payload,
    "table_shapes": table_shapes_df.to_dict(orient="records"),
}

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

handoff_manifest = {
    "schema_version": REPORT_SCHEMA_VERSION,
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "model_name": MODEL_NAME,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "pending_final_validation",
    "report_contract": {
        "expected_restoration_cases": EXPECTED_RESTORATION_CASES,
        "expected_report_cases": EXPECTED_REPORT_CASES,
        "expected_report_region": EXPECTED_REPORT_REGION,
        "expected_report_status": EXPECTED_REPORT_STATUS,
        "zero_controls_excluded_from_report_dataframe": True,
        "difference_map_manifest": project_relative_path(DIFFERENCE_MAP_MANIFEST_PATH),
    },
    "validation": {
        "final_validation": project_relative_path(FINAL_VALIDATION_OUTPUT_PATH),
        "final_validation_passed": False,
        "batch_validation_status": batch_status_payload,
    },
    "artifacts": {
        artifact_name: artifact_record(
            artifact_name,
            path,
            artifact_type=artifact_type,
            project_root=PROJECT_ROOT,
            required=required,
        )
        for artifact_name, (path, artifact_type, required) in FINAL_ARTIFACT_PATHS.items()
        if artifact_name != "handoff_manifest"
    },
    "table_shapes": table_shapes_df.to_dict(orient="records"),
    "downstream_notes": [
        "Use lama_report_dataframe.csv as the canonical non-zero LaMa report table.",
        "Use lama_report_selected_cases.csv and the figure manifest for qualitative examples.",
        "Use lama_restoration_evaluation_report.html as the human-readable synthesis report.",
        "Use this handoff manifest as the Notebook 19 completion contract.",
    ],
}

save_json(HANDOFF_MANIFEST_JSON_PATH, handoff_manifest)

required_csv_paths = {
    name: path
    for name, (path, artifact_type, required) in FINAL_ARTIFACT_PATHS.items()
    if artifact_type == "csv" and required and name != "final_validation"
}

required_csv_exists = {name: path.is_file() for name, path in required_csv_paths.items()}
required_csv_nonempty = {name: path_size(path) for name, path in required_csv_paths.items()}

selected_case_ids = set(selected_cases_df["case_id"].astype(str)) if "case_id" in selected_cases_df.columns else set()
report_case_ids = set(lama_report_df["case_id"].astype(str)) if "case_id" in lama_report_df.columns else set()

figure_manifest_exists = REPORT_FIGURE_MANIFEST_PATH.is_file()
html_report_exists = HTML_REPORT_OUTPUT_PATH.is_file()
stage_manifest_exists = STAGE_MANIFEST_JSON_PATH.is_file()
handoff_manifest_exists = HANDOFF_MANIFEST_JSON_PATH.is_file()
artifact_index_exists = FINAL_ARTIFACT_INDEX_OUTPUT_PATH.is_file()

final_validation_df = pd.DataFrame(
    [
        build_check(
            "all_prior_batch_validations_passed",
            batch_status_payload,
            "all true",
            all(batch_status_payload.values()),
            "one or more prior batch validation files is missing or failed",
        ),
        build_check(
            "report_dataframe_expected_rows",
            int(len(lama_report_df)),
            EXPECTED_REPORT_CASES,
            int(len(lama_report_df)) == EXPECTED_REPORT_CASES,
            "final report dataframe row count differs from expected non-zero case count",
        ),
        build_check(
            "report_case_ids_unique",
            int(lama_report_df["case_id"].nunique()) if "case_id" in lama_report_df.columns else 0,
            int(len(lama_report_df)),
            "case_id" in lama_report_df.columns and int(lama_report_df["case_id"].nunique()) == int(len(lama_report_df)),
            "report dataframe case IDs are missing or not unique",
        ),
        build_check(
            "selected_cases_subset_of_report",
            {"selected_cases": int(len(selected_case_ids)), "not_in_report": sorted(selected_case_ids - report_case_ids)[:10]},
            {"not_in_report": []},
            len(selected_case_ids - report_case_ids) == 0 and len(selected_case_ids) > 0,
            "one or more selected cases is not present in the report dataframe",
        ),
        build_check(
            "required_csv_outputs_exist",
            required_csv_exists,
            "all true",
            all(required_csv_exists.values()),
            "one or more required final CSV outputs is missing",
        ),
        build_check(
            "required_csv_outputs_nonempty",
            required_csv_nonempty,
            "all > 0",
            all(value > 0 for value in required_csv_nonempty.values()),
            "one or more required final CSV outputs is empty",
        ),
        build_check(
            "figure_manifest_exists",
            figure_manifest_exists,
            True,
            figure_manifest_exists and path_size(REPORT_FIGURE_MANIFEST_PATH) > 0,
            "figure manifest is missing or empty",
        ),
        build_check(
            "html_report_exists",
            html_report_exists,
            True,
            html_report_exists and path_size(HTML_REPORT_OUTPUT_PATH) > 0,
            "HTML report is missing or empty",
        ),
        build_check(
            "stage_manifest_exists",
            stage_manifest_exists,
            True,
            stage_manifest_exists and path_size(STAGE_MANIFEST_JSON_PATH) > 0,
            "stage manifest is missing or empty",
        ),
        build_check(
            "handoff_manifest_exists",
            handoff_manifest_exists,
            True,
            handoff_manifest_exists and path_size(HANDOFF_MANIFEST_JSON_PATH) > 0,
            "handoff manifest is missing or empty",
        ),
        build_check(
            "artifact_index_exists",
            artifact_index_exists,
            True,
            artifact_index_exists and path_size(FINAL_ARTIFACT_INDEX_OUTPUT_PATH) > 0,
            "artifact index is missing or empty",
        ),
    ]
)

final_validation_export_df = final_validation_df.copy()

for column in ["observed", "expected"]:
    final_validation_export_df[column] = final_validation_export_df[column].map(json_for_csv)

FINAL_VALIDATION_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final_validation_export_df.to_csv(FINAL_VALIDATION_OUTPUT_PATH, index=False)

final_validation_passed = bool(final_validation_df["passed"].astype(bool).all())
final_status = EXPECTED_REPORT_STATUS if final_validation_passed else "validation_failed"

stage_manifest["status"] = final_status
stage_manifest["final_validation"] = {
    "path": project_relative_path(FINAL_VALIDATION_OUTPUT_PATH),
    "passed": final_validation_passed,
    "checks_total": int(len(final_validation_df)),
    "checks_passed": int(final_validation_df["passed"].astype(bool).sum()),
}
stage_manifest["handoff_manifest"] = project_relative_path(HANDOFF_MANIFEST_JSON_PATH)
stage_manifest["artifact_index"] = project_relative_path(FINAL_ARTIFACT_INDEX_OUTPUT_PATH)
stage_manifest["batches"]["batch_6"]["validation_passed"] = final_validation_passed

save_json(STAGE_MANIFEST_JSON_PATH, stage_manifest)

handoff_manifest["status"] = final_status
handoff_manifest["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
handoff_manifest["validation"]["final_validation_passed"] = final_validation_passed
handoff_manifest["validation"]["checks_total"] = int(len(final_validation_df))
handoff_manifest["validation"]["checks_passed"] = int(final_validation_df["passed"].astype(bool).sum())
handoff_manifest["artifacts"]["handoff_manifest"] = artifact_record(
    "handoff_manifest",
    HANDOFF_MANIFEST_JSON_PATH,
    artifact_type="json",
    project_root=PROJECT_ROOT,
    required=True,
)

save_json(HANDOFF_MANIFEST_JSON_PATH, handoff_manifest)

artifact_index_df = build_artifact_index(FINAL_ARTIFACT_PATHS)
artifact_index_df.to_csv(FINAL_ARTIFACT_INDEX_OUTPUT_PATH, index=False)

display(batch_status_df)
display(final_validation_df)
display(artifact_index_df)

print("Final validation:", project_relative_path(FINAL_VALIDATION_OUTPUT_PATH))
print("Stage manifest:", project_relative_path(STAGE_MANIFEST_JSON_PATH))
print("Handoff manifest:", project_relative_path(HANDOFF_MANIFEST_JSON_PATH))
print("Artifact index:", project_relative_path(FINAL_ARTIFACT_INDEX_OUTPUT_PATH))

if not final_validation_passed:
    raise RuntimeError("Batch 6 final validation failed.")

print("Batch 6 complete.")

,batch,validation_path,validation_exists,validation_size_bytes,validation_passed
0,batch_1,outputs/19_lama_report_generation/validation/lama_report_generation_batch1_validation.csv,True,2432,True
1,batch_2,outputs/19_lama_report_generation/validation/lama_report_generation_batch2_validation.csv,True,2985,True
2,batch_3,outputs/19_lama_report_generation/validation/lama_report_generation_batch3_validation.csv,True,1084,True
3,batch_4,outputs/19_lama_report_generation/validation/lama_report_generation_batch4_validation.csv,True,1011,True
4,batch_5,outputs/19_lama_report_generation/validation/lama_report_generation_batch5_validation.csv,True,593,True


,check_name,observed,expected,passed,issue
0,all_prior_batch_validations_passed,"{'batch_1': True, 'batch_2': True, 'batch_3': True, 'batch_4': True, 'batch_5': True}",all true,True,
1,report_dataframe_expected_rows,355,355,True,
2,report_case_ids_unique,355,355,True,
3,selected_cases_subset_of_report,"{'selected_cases': 18, 'not_in_report': []}",{'not_in_report': []},True,
4,required_csv_outputs_exist,"{'report_dataframe': True, 'selected_cases': True, 'report_summaries': True, 'metric_correlations': True, 'metric_overview': True, 'report_status_cards': True, 'selected_case_r...",all true,True,
5,required_csv_outputs_nonempty,"{'report_dataframe': 1468035, 'selected_cases': 81682, 'report_summaries': 4428, 'metric_correlations': 681, 'metric_overview': 1173, 'report_status_cards': 223, 'selected_case...",all > 0,True,
6,figure_manifest_exists,True,True,True,
7,html_report_exists,True,True,True,
8,stage_manifest_exists,True,True,True,
9,handoff_manifest_exists,True,True,True,


,artifact,artifact_type,path,required,exists,size_bytes,sha256
0,report_dataframe,csv,outputs/19_lama_report_generation/metrics/lama_report_dataframe.csv,True,True,1468035,c032781d43c9cd27aeac949dad95fe80acf8680e6664b6c2df958ec345416669
1,selected_cases,csv,outputs/19_lama_report_generation/metrics/lama_report_selected_cases.csv,True,True,81682,314347ee1b207bae8f0fbf4da83b13291ae24cc08d8b1d0e0ea32bcfa6eaa2cf
2,report_summaries,csv,outputs/19_lama_report_generation/metrics/lama_report_summaries.csv,True,True,4428,1b690573ed7f624d076a0525c9abbea0a9425b283a4eec525a8128532b09dd99
3,metric_correlations,csv,outputs/19_lama_report_generation/metrics/lama_report_metric_correlations.csv,True,True,681,30e3e1a1eb7bd259afc220cf758c0678efcd5e3055754f3acc6face8217f6d6d
4,metric_overview,csv,outputs/19_lama_report_generation/metrics/lama_report_metric_overview.csv,True,True,1173,13a8b792c9b3f0201af89941ad8f246248942739b565f748744552729d274b92
5,report_status_cards,csv,outputs/19_lama_report_generation/metrics/lama_report_status_cards.csv,True,True,223,21de48205203f588170678af846e5a404e5aaa5c8845755d34d5e3fea0529bfe
6,report_findings,csv,outputs/19_lama_report_generation/metrics/lama_report_findings.csv,False,True,749,54a76b4f13b1ca131686b9aeaa8bbd6f4dc5c46109f5d6f81a591272ce012a3e
7,selected_case_report_table,csv,outputs/19_lama_report_generation/metrics/lama_report_selected_case_table.csv,True,True,8128,8c2dc8d0b166c017796b02635d4bf58094afc710ba2f10301d51858edc8f0b14
8,figure_manifest,csv,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_figure_manifest.csv,True,True,11685,4b997b88880998ac41424c0694984154880e219e0bbf303617e1a780a46b1e89
9,case_panel_manifest,csv,outputs/19_lama_report_generation/figures/report_diagnostics_lama/lama_report_selected_case_panel_manifest.csv,False,True,15948,56da89838a4fcd60839bd50c6e0cff59b117c861c4067c9f1d36df397dc0e114


Final validation: outputs/19_lama_report_generation/validation/lama_report_generation_validation.csv
Stage manifest: outputs/19_lama_report_generation/manifests/19_lama_report_generation_manifest.json
Handoff manifest: outputs/19_lama_report_generation/metrics/lama_report_handoff_manifest.json
Artifact index: outputs/19_lama_report_generation/manifests/19_lama_report_generation_artifact_index.csv
Batch 6 complete.
